In [1]:
import json

# Input
Input = '/home/martin/python_projects/Data_www/E12000003.geojson'

# Load and inspect first feature
with open(Input) as f:
    gj = json.load(f)

first_feature = gj['features'][0]['properties']

# Print as clean dictionary
for key, value in first_feature.items():
    print(f'{key}: {value}')

WD22CD: E05000976
WD22NM: Central
WD22NMW:  
LAD22CD: E08000016
LAD22NM: Barnsley
BNG_E: 435247
BNG_N: 406419
LONG: -1.46945
LAT: 53.55324
FID: 201
GlobalID: 1ed4e2a0-67e0-4c2b-87a1-dffc8bbe6877
LADCD_ACTIVE: E08000016
UTLACD: E08000016
CAUTHCD: E47000002
RGNCD: E12000003


In [2]:
import json

# Input
Input = '/home/martin/python_projects/Data_www/E12000003.geojson'
Output = '/home/martin/python_projects/Data_www/E12000003_clean.geojson'

# Columns to keep
keep_cols = ['WD22CD', 'WD22NM', 'LAD22CD', 'LAD22NM', 'LAT', 'LONG', 'RGNCD']

# Load GeoJSON
with open(Input) as f:
    gj = json.load(f)

# Strip junk from each feature
for feature in gj['features']:
    props = feature['properties']
    feature['properties'] = {k: props[k] for k in keep_cols}

# Save clean version
with open(Output, 'w') as f:
    json.dump(gj, f)

print(f'Done -- {len(gj["features"])} wards cleaned and saved to {Output}')

Done -- 411 wards cleaned and saved to /home/martin/python_projects/Data_www/E12000003_clean.geojson


In [1]:
#Scraoe Wakefield Election Results 2026
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Config
URL = 'https://www.wakefield.gov.uk/results2026'
OUTPUT = '/home/martin/python_projects/Data_www/Wakefield2026_raw.csv'

# Fetch page
response = requests.get(URL)
soup = BeautifulSoup(response.content, 'html.parser')

# Find all ward tables
tables = soup.find_all('table')

all_rows = []

for table in tables:
    # Get ward name from the first header cell
    header = table.find('th')
    if not header:
        continue
    ward_name = header.get_text(strip=True)

    rows = table.find_all('tr')
    for row in rows[1:]:  # Skip header row
        cols = row.find_all('td')
        if len(cols) < 3:
            continue

        candidate = cols[0].get_text(strip=True)
        party = cols[1].get_text(strip=True)
        votes = cols[2].get_text(strip=True)

        # Skip summary rows
        if 'Majority' in candidate or 'Spoiled' in candidate or 'Total' in candidate:
            continue

        # Flag elected
        elected = '(ELECTED)' in candidate
        candidate_clean = candidate.replace('(ELECTED)', '').strip()

        all_rows.append({
            'Ward': ward_name,
            'Candidate': candidate_clean,
            'Party': party,
            'Votes': votes,
            'Elected': elected,
            'Council': 'Wakefield',
            'Sub_Region': 'West Yorkshire',
            'RGNCD': 'E12000003'
        })

# Build dataframe
df = pd.DataFrame(all_rows)

# Clean votes column
df['Votes'] = pd.to_numeric(df['Votes'], errors='coerce')

# Normalise party names
party_map = {
    'Labour and Co-operative Party': 'Labour Party',
    'ReformUK - Changing Politics for Good': 'Reform UK',
    'The Conservative Party Candidate': 'Conservative Party',
    'Liberal Democrat Focus Team': 'Liberal Democrats'
}
df['Party'] = df['Party'].replace(party_map)

# Save
df.to_csv(OUTPUT, index=False)
print(f'Done -- {len(df)} rows saved to {OUTPUT}')
print(f'Wards found: {df["Ward"].nunique()}')
print()

# Dictionary preview -- first row
print('--- First row preview ---')
for key, value in df.iloc[0].to_dict().items():
    print(f'{key}: {value}')

Done -- 348 rows saved to /home/martin/python_projects/Data_www/Wakefield2026_raw.csv
Wards found: 21

--- First row preview ---
Ward: Ackworth, North Elmsall and Upton
Candidate: Atcheson, Kevin
Party: Reform UK
Votes: 2165.0
Elected: True
Council: Wakefield
Sub_Region: West Yorkshire
RGNCD: E12000003


In [2]:
# Fix votes to integer
df['Votes'] = df['Votes'].astype('Int64')

# Verify
print(df.iloc[0].to_dict())

{'Ward': 'Ackworth, North Elmsall and Upton', 'Candidate': 'Atcheson, Kevin', 'Party': 'Reform UK', 'Votes': 2165, 'Elected': True, 'Council': 'Wakefield', 'Sub_Region': 'West Yorkshire', 'RGNCD': 'E12000003'}


In [4]:
# Load Sheffield CSV
Sheffield_INPUT = '/home/martin/python_projects/Data_www/Sheffield_Local_Elections26_top3Candidates.csv'
df_sheffield = pd.read_csv(Sheffield_INPUT)

# Preview current schema
print('--- Sheffield current schema ---')
for key, value in df_sheffield.iloc[0].to_dict().items():
    print(f'{key}: {value}')

--- Sheffield current schema ---
Name: Tessa Louise Lupton
Description: Green Party
Votes: 4328
Ward: Ecclesall


In [5]:
# Rename columns to match unified schema
df_sheffield = df_sheffield.rename(columns={
    'Name': 'Candidate',
    'Description': 'Party'
})

# Add missing columns
df_sheffield['Elected'] = None
df_sheffield['Council'] = 'Sheffield'
df_sheffield['Sub_Region'] = 'South Yorkshire'
df_sheffield['RGNCD'] = 'E12000003'

# Normalise party names
df_sheffield['Party'] = df_sheffield['Party'].replace(party_map)

# Preview
print('--- Sheffield unified schema ---')
for key, value in df_sheffield.iloc[0].to_dict().items():
    print(f'{key}: {value}')

--- Sheffield unified schema ---
Candidate: Tessa Louise Lupton
Party: Green Party
Votes: 4328
Ward: Ecclesall
Elected: None
Council: Sheffield
Sub_Region: South Yorkshire
RGNCD: E12000003


In [6]:
# Merge Sheffield and Wakefield
df_unified = pd.concat([df_sheffield, df_wakefield], ignore_index=True)

# Verify
print(f'Total rows: {len(df_unified)}')
print(f'Councils: {df_unified["Council"].unique()}')
print(f'Sheffield rows: {len(df_unified[df_unified["Council"] == "Sheffield"])}')
print(f'Wakefield rows: {len(df_unified[df_unified["Council"] == "Wakefield"])}')
print()

# Save unified CSV
UNIFIED_OUTPUT = '/home/martin/python_projects/Data_www/Yorkshire_unified.csv'
df_unified.to_csv(UNIFIED_OUTPUT, index=False)
print(f'Saved to {UNIFIED_OUTPUT}')

NameError: name 'df_wakefield' is not defined

In [7]:
# Reload Wakefield from CSV
df_wakefield = pd.read_csv('/home/martin/python_projects/Data_www/Wakefield2026_raw.csv')

# Now merge
df_unified = pd.concat([df_sheffield, df_wakefield], ignore_index=True)

# Verify
print(f'Total rows: {len(df_unified)}')
print(f'Councils: {df_unified["Council"].unique()}')
print(f'Sheffield rows: {len(df_unified[df_unified["Council"] == "Sheffield"])}')
print(f'Wakefield rows: {len(df_unified[df_unified["Council"] == "Wakefield"])}')
print()

# Save unified CSV
UNIFIED_OUTPUT = '/home/martin/python_projects/Data_www/Yorkshire_unified.csv'
df_unified.to_csv(UNIFIED_OUTPUT, index=False)
print(f'Saved to {UNIFIED_OUTPUT}')

Total rows: 432
Councils: <StringArray>
['Sheffield', 'Wakefield']
Length: 2, dtype: str
Sheffield rows: 84
Wakefield rows: 348

Saved to /home/martin/python_projects/Data_www/Yorkshire_unified.csv


In [8]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Config
URL = 'https://www.barnsley.gov.uk/services/voting-and-elections/election-results/local-government-and-parish-council-elections-7-may-2026/'
OUTPUT = '/home/martin/python_projects/Data_www/Barnsley2026_raw.csv'

# Known local government wards -- to exclude parish council tables
LG_WARDS = [
    'Athersley and New Lodge ward',
    'Central ward',
    'Cudworth ward',
    'Darfield and Great Houghton ward',
    'Darton East ward',
    'Darton West ward',
    'Dearne North ward',
    'Dearne South ward',
    'Dodworth, Stainborough and Tankersley ward',
    'Hoyland Milton ward',
    'Kingstone ward',
    'Monk Bretton ward',
    'North East ward',
    'Old Town ward',
    'Penistone East ward',
    'Penistone West ward',
    'Rockingham ward',
    'Royston ward',
    'Stairfoot ward',
    'Wombwell ward',
    'Worsbrough ward'
]

# Fetch page
response = requests.get(URL)
soup = BeautifulSoup(response.content, 'html.parser')

all_rows = []
current_ward = None

# Walk through h3 headings and tables together
for tag in soup.find_all(['h3', 'table']):
    if tag.name == 'h3':
        heading = tag.get_text(strip=True)
        if heading in LG_WARDS:
            current_ward = heading.replace(' ward', '').replace(' Ward', '')
        else:
            current_ward = None

    elif tag.name == 'table' and current_ward:
        rows = tag.find_all('tr')
        for row in rows[1:]:  # Skip header
            cols = row.find_all('td')
            if len(cols) < 3:
                continue

            candidate = cols[0].get_text(strip=True)
            party = cols[1].get_text(strip=True)
            votes_raw = cols[2].get_text(strip=True)

            # Skip summary rows
            if any(x in candidate for x in ['Majority', 'Spoiled', 'Total']):
                continue

            # Elected = row is bold
            elected = bool(row.find('strong') or row.find('b'))

            # Clean votes -- strip commas and ELECTED text
            votes_clean = votes_raw.replace(',', '').replace('- ELECTED', '').strip()

            all_rows.append({
                'Ward': current_ward,
                'Candidate': candidate,
                'Party': party,
                'Votes': votes_clean,
                'Elected': elected,
                'Council': 'Barnsley',
                'Sub_Region': 'South Yorkshire',
                'RGNCD': 'E12000003'
            })

# Build dataframe
df = pd.DataFrame(all_rows)

# Clean votes to integer
df['Votes'] = pd.to_numeric(df['Votes'], errors='coerce').astype('Int64')

# Normalise party names
party_map = {
    'The Green Party': 'Green Party',
    'Labour and Co-operative Party': 'Labour Party',
    'ReformUK - Changing Politics for Good': 'Reform UK',
    'Conservative Party Candidate': 'Conservative Party'
}
df['Party'] = df['Party'].replace(party_map)

# Save
df.to_csv(OUTPUT, index=False)
print(f'Done -- {len(df)} rows saved to {OUTPUT}')
print(f'Wards found: {df["Ward"].nunique()}')
print()

# Dictionary preview -- first row
print('--- First row preview ---')
for key, value in df.iloc[0].to_dict().items():
    print(f'{key}: {value}')

KeyError: 'Votes'

In [10]:
# Diagnose -- find what tags the ward headings actually use
response = requests.get(URL)
soup = BeautifulSoup(response.content, 'html.parser')

# Print all headings found
for tag in soup.find_all(['h2', 'h3', 'h4']):
    text = tag.get_text(strip=True)
    if 'ward' in text.lower() or 'Ward' in text:
        print(f'{tag.name}: {text}')

In [11]:
# Find where ward names actually appear in the HTML
response = requests.get(URL)
soup = BeautifulSoup(response.content, 'html.parser')

# Search all tags for known ward name
for tag in soup.find_all(True):
    if 'Athersley' in tag.get_text():
        print(f'Tag: {tag.name}, Class: {tag.get("class")}, Text: {tag.get_text(strip=True)[:80]}')

In [12]:
import requests
from bs4 import BeautifulSoup

# Config
URL = 'https://www.barnsley.gov.uk/services/voting-and-elections/election-results/local-government-and-parish-council-elections-7-may-2026/'

# Fresh fetch
response = requests.get(URL)
print(f'Status code: {response.status_code}')
print(f'Page length: {len(response.content)} bytes')

# Parse
soup = BeautifulSoup(response.content, 'html.parser')

# Search all tags for Athersley
for tag in soup.find_all(True):
    if 'Athersley' in tag.get_text(strip=True):
        print(f'Tag: {tag.name} | Class: {tag.get("class")} | Text: {tag.get_text(strip=True)[:80]}')

Status code: 403
Page length: 179 bytes


In [13]:
import requests
from bs4 import BeautifulSoup

URL = 'https://www.barnsley.gov.uk/services/voting-and-elections/election-results/local-government-and-parish-council-elections-7-may-2026/'

# Mimic a real browser
headers = {
    'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

response = requests.get(URL, headers=headers)
print(f'Status code: {response.status_code}')
print(f'Page length: {len(response.content)} bytes')

soup = BeautifulSoup(response.content, 'html.parser')

# Search all tags for Athersley
for tag in soup.find_all(True):
    if 'Athersley' in tag.get_text(strip=True):
        print(f'Tag: {tag.name} | Class: {tag.get("class")} | Text: {tag.get_text(strip=True)[:80]}')

Status code: 200
Page length: 119758 bytes
Tag: html | Class: ['no-js'] | Text: Local government and parish council elections - 7 May 2026Skip to contentMenuSea
Tag: body | Class: None | Text: Skip to contentMenuSearchBSLTranslateAlertsImportant alertsDisruptions to bin co
Tag: div | Class: ['container', 'container--main'] | Text: You are here:HomeServicesVoting and electionsElection resultsLocal government an
Tag: div | Class: ['main'] | Text: You are here:HomeServicesVoting and electionsElection resultsLocal government an
Tag: article | Class: None | Text: You are here:HomeServicesVoting and electionsElection resultsLocal government an
Tag: div | Class: ['container', 'container--page'] | Text: You are here:HomeServicesVoting and electionsElection resultsLocal government an
Tag: div | Class: ['container', 'container--grid'] | Text: Go backBack to Election resultsLocal government and parish council elections - 7
Tag: div | Class: ['container', 'container--content', 'container--content-

In [14]:
import requests
from bs4 import BeautifulSoup
import pandas as pd

# Config
URL = 'https://www.barnsley.gov.uk/services/voting-and-elections/election-results/local-government-and-parish-council-elections-7-may-2026/'
OUTPUT = '/home/martin/python_projects/Data_www/Barnsley2026_raw.csv'

# Browser header
headers = {
    'User-Agent': 'Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36'
}

# Known local government wards
LG_WARDS = [
    'Athersley and New Lodge ward',
    'Central ward',
    'Cudworth ward',
    'Darfield and Great Houghton ward',
    'Darton East ward',
    'Darton West ward',
    'Dearne North ward',
    'Dearne South ward',
    'Dodworth, Stainborough and Tankersley ward',
    'Hoyland Milton ward',
    'Kingstone ward',
    'Monk Bretton ward',
    'North East ward',
    'Old Town ward',
    'Penistone East ward',
    'Penistone West ward',
    'Rockingham ward',
    'Royston ward',
    'Stairfoot ward',
    'Wombwell ward',
    'Worsbrough ward'
]

# Fetch page
response = requests.get(URL, headers=headers)
soup = BeautifulSoup(response.content, 'html.parser')

all_rows = []
current_ward = None

# Walk through h3 headings and tables together
for tag in soup.find_all(['h3', 'table']):
    if tag.name == 'h3':
        heading = tag.get_text(strip=True)
        if heading in LG_WARDS:
            current_ward = heading.replace(' ward', '').replace(' Ward', '')
        else:
            current_ward = None

    elif tag.name == 'table' and current_ward:
        rows = tag.find_all('tr')
        for row in rows[1:]:
            cols = row.find_all('td')
            if len(cols) < 3:
                continue

            candidate = cols[0].get_text(strip=True)
            party = cols[1].get_text(strip=True)
            votes_raw = cols[2].get_text(strip=True)

            # Skip summary rows
            if any(x in candidate for x in ['Majority', 'Spoiled', 'Total']):
                continue

            # Elected = row is bold
            elected = bool(row.find('strong') or row.find('b'))

            # Clean votes
            votes_clean = votes_raw.replace(',', '').replace('- ELECTED', '').strip()

            all_rows.append({
                'Ward': current_ward,
                'Candidate': candidate,
                'Party': party,
                'Votes': votes_clean,
                'Elected': elected,
                'Council': 'Barnsley',
                'Sub_Region': 'South Yorkshire',
                'RGNCD': 'E12000003'
            })

# Build dataframe
df = pd.DataFrame(all_rows)

# Clean votes to integer
df['Votes'] = pd.to_numeric(df['Votes'], errors='coerce').astype('Int64')

# Normalise party names
party_map = {
    'The Green Party': 'Green Party',
    'Labour and Co-operative Party': 'Labour Party',
    'ReformUK - Changing Politics for Good': 'Reform UK',
    'Conservative Party Candidate': 'Conservative Party'
}
df['Party'] = df['Party'].replace(party_map)

# Save
df.to_csv(OUTPUT, index=False)
print(f'Done -- {len(df)} rows saved to {OUTPUT}')
print(f'Wards found: {df["Ward"].nunique()}')
print()

# Dictionary preview -- first row
print('--- First row preview ---')
for key, value in df.iloc[0].to_dict().items():
    print(f'{key}: {value}')

Done -- 288 rows saved to /home/martin/python_projects/Data_www/Barnsley2026_raw.csv
Wards found: 21

--- First row preview ---
Ward: Athersley and New Lodge
Candidate: Andrew Allerton
Party: Conservative Party
Votes: 113
Elected: False
Council: Barnsley
Sub_Region: South Yorkshire
RGNCD: E12000003


In [15]:
# Load existing unified and Barnsley
df_unified = pd.read_csv('/home/martin/python_projects/Data_www/Yorkshire_unified.csv')
df_barnsley = pd.read_csv('/home/martin/python_projects/Data_www/Barnsley2026_raw.csv')

# Append
df_unified = pd.concat([df_unified, df_barnsley], ignore_index=True)

# Verify
print(f'Total rows: {len(df_unified)}')
print(f'Councils: {df_unified["Council"].unique()}')
print(f'Sheffield rows: {len(df_unified[df_unified["Council"] == "Sheffield"])}')
print(f'Wakefield rows: {len(df_unified[df_unified["Council"] == "Wakefield"])}')
print(f'Barnsley rows: {len(df_unified[df_unified["Council"] == "Barnsley"])}')

# Save
df_unified.to_csv('/home/martin/python_projects/Data_www/Yorkshire_unified.csv', index=False)
print(f'\nSaved -- Yorkshire_unified.csv updated!')

Total rows: 720
Councils: <StringArray>
['Sheffield', 'Wakefield', 'Barnsley']
Length: 3, dtype: str
Sheffield rows: 84
Wakefield rows: 348
Barnsley rows: 288

Saved -- Yorkshire_unified.csv updated!


In [16]:
import geopandas as gpd
import pandas as pd
import folium
from folium.plugins import GroupedLayerControl
from jinja2 import Template
from branca.element import Element

# File paths
GEOJSON = '/home/martin/python_projects/Data_www/E12000003_clean.geojson'
CSV = '/home/martin/python_projects/Data_www/Yorkshire_unified.csv'

# Load both
gdf = gpd.read_file(GEOJSON)
df = pd.read_csv(CSV)

print(f'GeoJSON wards: {len(gdf)}')
print(f'CSV rows: {len(df)}')
print(f'Councils in CSV: {df["Council"].unique()}')

GeoJSON wards: 411
CSV rows: 720
Councils in CSV: <StringArray>
['Sheffield', 'Wakefield', 'Barnsley']
Length: 3, dtype: str


In [17]:
# Party colours
party_colours = {
    'Green Party':        '#4CAF50',
    'Liberal Democrats':  '#FAA61A',
    'Labour Party':       '#E4003B',
    'Reform UK':          '#12B6CF',
    'Independent':        '#888888',
    'Conservative Party': '#1D4F8C'
}

# Fix GeoJSON ward names
gdf['Ward'] = gdf['WD22NM'].str.strip()

# Fix CSV ward names
df['Ward'] = df['Ward'].str.replace('\xa0', ' ', regex=False).str.strip()

# Check mismatches for our three councils only
our_councils = ['Sheffield', 'Wakefield', 'Barnsley']
df_ours = df[df['Council'].isin(our_councils)]

geojson_wards = set(gdf['Ward'])
csv_wards = set(df_ours['Ward'])

print('In CSV but NOT in GeoJSON:')
print(csv_wards - geojson_wards)
print()
print('Matches found:', len(csv_wards & geojson_wards))

In CSV but NOT in GeoJSON:
{'Athersley and New Lodge', 'Darfield and Great Houghton', 'Knottingley and Ferrybridge', 'Dodworth, Stainborough and Tankersley'}

Matches found: 66


In [18]:
# Find the GeoJSON equivalents
search_terms = ['Athersley', 'Darfield', 'Knottingley', 'Dodworth']

for term in search_terms:
    matches = gdf[gdf['Ward'].str.contains(term, case=False)]['Ward'].tolist()
    print(f'{term}: {matches}')

Athersley: []
Darfield: ['Darfield']
Knottingley: ['Knottingley']
Dodworth: ['Dodworth']


In [19]:
# Check all Barnsley wards in GeoJSON
barnsley_wards = gdf[gdf['LAD22NM'] == 'Barnsley']['Ward'].tolist()
print('Barnsley wards in GeoJSON:')
for w in sorted(barnsley_wards):
    print(f'  {w}')

Barnsley wards in GeoJSON:
  Central
  Cudworth
  Darfield
  Darton East
  Darton West
  Dearne North
  Dearne South
  Dodworth
  Hoyland Milton
  Kingstone
  Monk Bretton
  North East
  Old Town
  Penistone East
  Penistone West
  Rockingham
  Royston
  St Helens
  Stairfoot
  Wombwell
  Worsbrough


In [22]:
# Manual ward name fixes for CSV to match GeoJSON
ward_name_fixes = {
    'Athersley and New Lodge': 'St Helens',
    'Darfield and Great Houghton': 'Darfield',
    'Dodworth, Stainborough and Tankersley': 'Dodworth',
    'Knottingley and Ferrybridge': 'Knottingley'
}

df['Ward'] = df['Ward'].replace(ward_name_fixes)

# Verify -- recheck mismatches
df_ours = df[df['Council'].isin(our_councils)]
csv_wards = set(df_ours['Ward'])
geojson_wards = set(gdf['Ward'])

print('Mismatches remaining:')
print(csv_wards - geojson_wards)
print()
print('Matches found:', len(csv_wards & geojson_wards))

Mismatches remaining:
set()

Matches found: 70


In [23]:
# Normalise party names
party_map = {
    'Labour and Co-operative Party': 'Labour Party',
    'ReformUK - Changing Politics for Good': 'Reform UK',
    'The Conservative Party Candidate': 'Conservative Party',
    'Conservative Party Candidate': 'Conservative Party',
    'Liberal Democrat Focus Team': 'Liberal Democrats'
}
df['Party'] = df['Party'].replace(party_map)

# Get winner per ward
winners = df.loc[df.groupby('Ward')['Votes'].idxmax()].reset_index(drop=True)

# Merge winners with GeoJSON
gdf_merged = gdf.merge(winners[['Ward', 'Party', 'Votes']], on='Ward', how='left')
gdf_merged['colour'] = gdf_merged['Party'].map(party_colours).fillna('#cccccc')

# Build ward results dict for tooltips
ward_results = {}
for ward, group in df.groupby('Ward'):
    group_sorted = group.sort_values('Votes', ascending=False).head(3).reset_index(drop=True)
    ward_results[ward] = group_sorted[['Party', 'Votes']].to_dict('records')

# Sanity check
print(f'Merged wards: {len(gdf_merged)}')
print(f'Wards with results: {gdf_merged["Party"].notna().sum()}')
print(f'Wards without results: {gdf_merged["Party"].isna().sum()}')

Merged wards: 411
Wards with results: 73
Wards without results: 338


In [24]:
# Special notes
special_notes = {
    'Firth Park': '⚠️ Two seats contested — Reform UK & Labour Party each won one seat.',
    'Wakefield East': '⚠️ Two seats contested — Labour Party & Reform UK each won one seat.'
}

# Base map
m = folium.Map(location=[53.6, -1.5], zoom_start=10, tiles='CartoDB positron')

# Inject CSS
meta = Element("""
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<style>
    .leaflet-tooltip {
        font-size: 14px !important;
        min-width: 260px !important;
        padding: 10px !important;
        pointer-events: none;
    }
</style>
""")
m.get_root().header.add_child(meta)

# One FeatureGroup per council
councils = ['Sheffield', 'Barnsley', 'Wakefield']
council_groups = {}

for council in councils:
    council_groups[council] = folium.FeatureGroup(name=council, show=True)

# Grey layer for uncontested wards
grey_group = folium.FeatureGroup(name='Uncontested wards', show=True)

for _, row in gdf_merged.iterrows():
    ward = row['Ward']
    party = row['Party']
    colour = row['colour']

    # Determine which group
    if pd.isna(party):
        target_group = grey_group
        tooltip_html = f'<div style="font-family:Arial; padding:8px;"><b>{ward}</b><br><span style="color:#888;">No 2026 election data</span></div>'
    else:
        # Find council for this ward
        council = df[df['Ward'] == ward]['Council'].iloc[0] if ward in df['Ward'].values else None
        target_group = council_groups.get(council, grey_group)

        results = ward_results.get(ward, [])
        max_votes = max(r['Votes'] for r in results) if results else 1

        bars_html = ""
        for r in results:
            c = party_colours.get(r['Party'], '#888888')
            bar_width = int((r['Votes'] / max_votes) * 200)
            bars_html += f"""
            <div style="margin:6px 0; font-size:13px;">
                <div style="margin-bottom:2px;">{r['Party']}: <b>{r['Votes']}</b></div>
                <div style="background:{c}; width:{bar_width}px; height:14px; border-radius:3px;"></div>
            </div>
            """

        note = special_notes.get(ward, '')
        note_html = f'<div style="font-size:11px; color:#888; margin-top:8px;">{note}</div>' if note else ''

        tooltip_html = f"""
        <div style="font-family:Arial; min-width:250px; padding:8px;">
            <b style="font-size:15px;">{ward}</b><br>
            <span style="font-size:12px; color:#888;">{council}</span><br><br>
            {bars_html}
            {note_html}
        </div>
        """

    folium.GeoJson(
        row['geometry'],
        style_function=lambda x, colour=colour: {
            'fillColor': colour,
            'color': 'white',
            'weight': 1.5,
            'fillOpacity': 0.5 if colour != '#cccccc' else 0.15
        },
        tooltip=folium.Tooltip(tooltip_html, sticky=True)
    ).add_to(target_group)

# Add all groups to map
for group in council_groups.values():
    group.add_to(m)
grey_group.add_to(m)

# Layer control
folium.LayerControl(collapsed=False).add_to(m)

print('Map built -- adding legend next')

Map built -- adding legend next


In [31]:
# Special notes
special_notes = {
    'Firth Park': '⚠️ Two seats contested — Reform UK & Labour Party each won one seat.',
    'Wakefield East': '⚠️ Two seats contested — Labour Party & Reform UK each won one seat.'
}

# Base map
m = folium.Map(location=[53.6, -1.5], zoom_start=10, tiles='CartoDB positron')

# Inject CSS
meta = Element("""
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<style>
    .leaflet-tooltip {
        font-size: 14px !important;
        min-width: 260px !important;
        padding: 10px !important;
        pointer-events: none;
    }
</style>
""")
m.get_root().header.add_child(meta)

# One FeatureGroup per council
councils = ['Sheffield', 'Barnsley', 'Wakefield']
council_groups = {}
for council in councils:
    council_groups[council] = folium.FeatureGroup(name=council, show=True)

grey_group = folium.FeatureGroup(name='No 2026 data', show=False)

for _, row in gdf_merged.iterrows():
    ward = row['Ward']
    party = row['Party']
    colour = row['colour']

    if pd.isna(party):
        target_group = grey_group
        tooltip_html = f'<div style="font-family:Arial; padding:8px;"><b>{ward}</b><br><span style="color:#888;">No 2026 election data</span></div>'
    else:
        council = df[df['Ward'] == ward]['Council'].iloc[0] if ward in df['Ward'].values else None
        target_group = council_groups.get(council, grey_group)

        results = ward_results.get(ward, [])
        max_votes = max(r['Votes'] for r in results) if results else 1

        bars_html = ""
        for r in results:
            c = party_colours.get(r['Party'], '#888888')
            bar_width = int((r['Votes'] / max_votes) * 200)
            bars_html += f"""
            <div style="margin:6px 0; font-size:13px;">
                <div style="margin-bottom:2px;">{r['Party']}: <b>{r['Votes']}</b></div>
                <div style="background:{c}; width:{bar_width}px; height:14px; border-radius:3px;"></div>
            </div>
            """

        note = special_notes.get(ward, '')
        note_html = f'<div style="font-size:11px; color:#888; margin-top:8px;">{note}</div>' if note else ''

        tooltip_html = f"""
        <div style="font-family:Arial; min-width:250px; padding:8px;">
            <b style="font-size:15px;">{ward}</b><br>
            <span style="font-size:12px; color:#888;">{council}</span><br><br>
            {bars_html}
            {note_html}
        </div>
        """

    folium.GeoJson(
        row['geometry'],
        style_function=lambda x, colour=colour: {
            'fillColor': colour,
            'color': 'white',
            'weight': 1.5,
            'fillOpacity': 0.5 if colour != '#cccccc' else 0.15
        },
        tooltip=folium.Tooltip(tooltip_html, sticky=True)
    ).add_to(target_group)

# Add all groups to map
for group in council_groups.values():
    group.add_to(m)
grey_group.add_to(m)

# Layer control
folium.LayerControl(collapsed=False).add_to(m)

# Legend -- single clean block
legend_html = """
<div style="position: fixed; bottom: 0; left: 0; right: 0; z-index: 1000; font-family: Arial;">

    <!-- Toggle button -->
    <div style="text-align:center;">
        <button onclick="
            var panel = document.getElementById('legend-panel');
            var btn = document.getElementById('legend-toggle-btn');
            if(panel.style.display === 'none') {
                panel.style.display = 'block';
                btn.innerHTML = '▼ Hide legend';
            } else {
                panel.style.display = 'none';
                btn.innerHTML = '▲ Show legend';
            }"
            id="legend-toggle-btn"
            style="background:white; border:1px solid #ccc; border-radius:6px 6px 0 0;
                   padding:4px 16px; cursor:pointer; font-size:12px;
                   box-shadow: 0 -2px 6px rgba(0,0,0,0.15);">
            ▼ Hide legend
        </button>
    </div>

    <!-- Legend panel -->
    <div id="legend-panel" style="background-color: white; padding: 8px 20px;
            box-shadow: 0 -2px 6px rgba(0,0,0,0.2); text-align: center;">

        <!-- Line 1 -- Parties -->
        <div style="white-space: nowrap; overflow-x: auto; font-size: 13px; margin-bottom: 4px;">
            {% for party, colour in parties.items() %}
            <span style="display:inline-block; margin: 0 10px;">
                <span style="background:{{ colour }}; width:12px; height:12px;
                             display:inline-block; border-radius:3px; margin-right:4px;
                             vertical-align:middle;"></span>
                {{ party }}
            </span>
            {% endfor %}
            <span style="display:inline-block; margin: 0 10px;">
                <span style="background:#cccccc; width:12px; height:12px;
                             display:inline-block; border-radius:3px; margin-right:4px;
                             vertical-align:middle;"></span>
                No 2026 data
            </span>
        </div>

        <!-- Line 2 -- Source -->
        <div style="font-size:11px; color:#555; margin-bottom: 3px;">
            Source: Official results from Sheffield, Barnsley &amp; Wakefield Metropolitan Borough Councils
        </div>

        <!-- Line 3 -- Creators -->
        <div style="font-size:11px; color:#888;">
            Built by encephalon_zest · Co-authored with Claude (Anthropic) · 2026
        </div>

    </div>
</div>
"""
legend = Template(legend_html).render(parties=party_colours)
m.get_root().html.add_child(folium.Element(legend))

# Save
OUTPUT = '/home/martin/python_projects/Data_www/Yorkshire_map.html'
m.save(OUTPUT)
print('Done -- refresh your browser!')

Done -- refresh your browser!


In [32]:
# Check duplicate ward names across councils in GeoJSON
duplicate_wards = ['City', 'Richmond', 'Central']

for ward in duplicate_wards:
    matches = gdf[gdf['Ward'] == ward][['Ward', 'LAD22NM']].values.tolist()
    print(f'{ward}:')
    for m in matches:
        print(f'  {m[1]}')
    print()

City:
  Bradford
  Sheffield

Richmond:
  Sheffield
  Richmondshire

Central:
  Barnsley
  Kingston upon Hull, City of



In [33]:
# Council name lookup matching GeoJSON LAD22NM values
council_to_lad = {
    'Sheffield': 'Sheffield',
    'Barnsley': 'Barnsley',
    'Wakefield': 'Wakefield'
}

df['LAD22NM'] = df['Council'].map(council_to_lad)

# Rebuild winners with LAD22NM
winners = df.loc[df.groupby(['Ward', 'LAD22NM'])['Votes'].idxmax()].reset_index(drop=True)

# Merge on BOTH Ward and LAD22NM
gdf_merged = gdf.merge(winners[['Ward', 'LAD22NM', 'Party', 'Votes']], 
                        on=['Ward', 'LAD22NM'], 
                        how='left')

gdf_merged['colour'] = gdf_merged['Party'].map(party_colours).fillna('#cccccc')

# Verify -- check the problematic wards
for ward in ['City', 'Richmond', 'Central']:
    matches = gdf_merged[gdf_merged['Ward'] == ward][['Ward', 'LAD22NM', 'Party']].values.tolist()
    print(f'{ward}:')
    for m in matches:
        print(f'  {m[1]} -- {m[2]}')
    print()

City:
  Bradford -- nan
  Sheffield -- Green Party

Richmond:
  Sheffield -- Reform UK
  Richmondshire -- nan

Central:
  Barnsley -- Labour Party
  Kingston upon Hull, City of -- nan



In [34]:
# Rebuild ward results dict keyed on Ward + LAD22NM
ward_results = {}
for (ward, lad), group in df.groupby(['Ward', 'LAD22NM']):
    group_sorted = group.sort_values('Votes', ascending=False).head(3).reset_index(drop=True)
    ward_results[(ward, lad)] = group_sorted[['Party', 'Votes']].to_dict('records')

# Sanity check
print(ward_results[('City', 'Sheffield')])
print(ward_results[('Central', 'Barnsley')])

[{'Party': 'Green Party', 'Votes': 2524.0}, {'Party': 'Labour Party', 'Votes': 434.0}, {'Party': 'Reform UK', 'Votes': 239.0}]
[{'Party': 'Labour Party', 'Votes': 1208.0}, {'Party': 'Labour Party', 'Votes': 1137.0}, {'Party': 'Labour Party', 'Votes': 1089.0}]


In [35]:
# Rebuild ward results dict -- top 3 unique parties by highest vote getter
ward_results = {}
for (ward, lad), group in df.groupby(['Ward', 'LAD22NM']):
    # Get best result per party
    best_per_party = group.groupby('Party')['Votes'].max().reset_index()
    best_per_party = best_per_party.sort_values('Votes', ascending=False).head(3).reset_index(drop=True)
    ward_results[(ward, lad)] = best_per_party[['Party', 'Votes']].to_dict('records')

# Sanity check
print('City Sheffield:')
print(ward_results[('City', 'Sheffield')])
print()
print('Central Barnsley:')
print(ward_results[('Central', 'Barnsley')])

City Sheffield:
[{'Party': 'Green Party', 'Votes': 2524.0}, {'Party': 'Labour Party', 'Votes': 434.0}, {'Party': 'Reform UK', 'Votes': 239.0}]

Central Barnsley:
[{'Party': 'Labour Party', 'Votes': 1208.0}, {'Party': 'Reform UK', 'Votes': 1033.0}, {'Party': 'Green Party', 'Votes': 286.0}]


In [36]:
# Special notes
special_notes = {
    'Firth Park': '⚠️ Two seats contested — Reform UK & Labour Party each won one seat.',
    'Wakefield East': '⚠️ Two seats contested — Labour Party & Reform UK each won one seat.'
}

# Base map
m = folium.Map(location=[53.6, -1.5], zoom_start=10, tiles='CartoDB positron')

# Inject CSS
meta = Element("""
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<style>
    .leaflet-tooltip {
        font-size: 14px !important;
        min-width: 260px !important;
        padding: 10px !important;
        pointer-events: none;
    }
</style>
""")
m.get_root().header.add_child(meta)

# One FeatureGroup per council
councils = ['Sheffield', 'Barnsley', 'Wakefield']
council_groups = {}
for council in councils:
    council_groups[council] = folium.FeatureGroup(name=council, show=True)

grey_group = folium.FeatureGroup(name='No 2026 data', show=False)

for _, row in gdf_merged.iterrows():
    ward = row['Ward']
    lad = row['LAD22NM']
    party = row['Party']
    colour = row['colour']

    if pd.isna(party):
        target_group = grey_group
        tooltip_html = f'<div style="font-family:Arial; padding:8px;"><b>{ward}</b><br><span style="color:#888;">No 2026 election data</span></div>'
    else:
        council = df[df['Ward'] == ward]['Council'].iloc[0] if ward in df['Ward'].values else None
        target_group = council_groups.get(council, grey_group)

        results = ward_results.get((ward, lad), [])
        max_votes = max(r['Votes'] for r in results) if results else 1

        bars_html = ""
        for r in results:
            c = party_colours.get(r['Party'], '#888888')
            bar_width = int((r['Votes'] / max_votes) * 200)
            bars_html += f"""
            <div style="margin:6px 0; font-size:13px;">
                <div style="margin-bottom:2px;">{r['Party']}: <b>{int(r['Votes'])}</b></div>
                <div style="background:{c}; width:{bar_width}px; height:14px; border-radius:3px;"></div>
            </div>
            """

        note = special_notes.get(ward, '')
        note_html = f'<div style="font-size:11px; color:#888; margin-top:8px;">{note}</div>' if note else ''

        tooltip_html = f"""
        <div style="font-family:Arial; min-width:250px; padding:8px;">
            <b style="font-size:15px;">{ward}</b><br>
            <span style="font-size:12px; color:#888;">{council}</span><br><br>
            {bars_html}
            {note_html}
        </div>
        """

    folium.GeoJson(
        row['geometry'],
        style_function=lambda x, colour=colour: {
            'fillColor': colour,
            'color': 'white',
            'weight': 1.5,
            'fillOpacity': 0.5 if colour != '#cccccc' else 0.15
        },
        tooltip=folium.Tooltip(tooltip_html, sticky=True)
    ).add_to(target_group)

# Add all groups to map
for group in council_groups.values():
    group.add_to(m)
grey_group.add_to(m)

# Layer control
folium.LayerControl(collapsed=False).add_to(m)

# Legend
legend_html = """
<div style="position: fixed; bottom: 0; left: 0; right: 0; z-index: 1000; font-family: Arial;">
    <div style="text-align:center;">
        <button onclick="
            var panel = document.getElementById('legend-panel');
            var btn = document.getElementById('legend-toggle-btn');
            if(panel.style.display === 'none') {
                panel.style.display = 'block';
                btn.innerHTML = '▼ Hide legend';
            } else {
                panel.style.display = 'none';
                btn.innerHTML = '▲ Show legend';
            }"
            id="legend-toggle-btn"
            style="background:white; border:1px solid #ccc; border-radius:6px 6px 0 0;
                   padding:4px 16px; cursor:pointer; font-size:12px;
                   box-shadow: 0 -2px 6px rgba(0,0,0,0.15);">
            ▼ Hide legend
        </button>
    </div>
    <div id="legend-panel" style="background-color: white; padding: 8px 20px;
            box-shadow: 0 -2px 6px rgba(0,0,0,0.2); text-align: center;">
        <div style="white-space: nowrap; overflow-x: auto; font-size: 13px; margin-bottom: 4px;">
            {% for party, colour in parties.items() %}
            <span style="display:inline-block; margin: 0 10px;">
                <span style="background:{{ colour }}; width:12px; height:12px;
                             display:inline-block; border-radius:3px; margin-right:4px;
                             vertical-align:middle;"></span>
                {{ party }}
            </span>
            {% endfor %}
            <span style="display:inline-block; margin: 0 10px;">
                <span style="background:#cccccc; width:12px; height:12px;
                             display:inline-block; border-radius:3px; margin-right:4px;
                             vertical-align:middle;"></span>
                No 2026 data
            </span>
        </div>
        <div style="font-size:11px; color:#555; margin-bottom: 3px;">
            Source: Official results from Sheffield, Barnsley &amp; Wakefield Metropolitan Borough Councils
        </div>
        <div style="font-size:11px; color:#888;">
            Built by encephalon_zest · Co-authored with Claude (Anthropic) · 2026
        </div>
    </div>
</div>
"""

legend = Template(legend_html).render(parties=party_colours)
m.get_root().html.add_child(folium.Element(legend))

OUTPUT = '/home/martin/python_projects/Data_www/Yorkshire_map.html'
m.save(OUTPUT)
print('Done -- refresh your browser!')

Done -- refresh your browser!


In [37]:
# Sheffield full council composition -- supplied
sheffield_composition = {
    'Labour Party':      25,
    'Liberal Democrats': 22,
    'Green Party':       20,
    'Reform UK':         13,
    'Independent':        4
}

# Barnsley and Wakefield -- calculate from Elected column
for council in ['Barnsley', 'Wakefield']:
    df_council = df[(df['Council'] == council) & (df['Elected'] == True)]
    composition = df_council.groupby('Party')['Party'].count().sort_values(ascending=False)
    print(f'{council} composition:')
    print(composition)
    print()

Barnsley composition:
Party
Reform UK            42
Labour Party         11
Liberal Democrats     8
Independent           2
Name: Party, dtype: int64

Wakefield composition:
Party
Reform UK             58
Liberal Democrats      2
Conservative Party     1
Labour Party           1
Green Party            1
Name: Party, dtype: int64



In [38]:
# Build compositions dict
council_compositions = {
    'Sheffield': sheffield_composition,
    'Barnsley': dict(df[(df['Council'] == 'Barnsley') & (df['Elected'] == True)].groupby('Party')['Party'].count()),
    'Wakefield': dict(df[(df['Council'] == 'Wakefield') & (df['Elected'] == True)].groupby('Party')['Party'].count())
}

# Calculate council centroids from GeoJSON
council_centroids = {}
for council, lad in [('Sheffield', 'Sheffield'), ('Barnsley', 'Barnsley'), ('Wakefield', 'Wakefield')]:
    wards = gdf[gdf['LAD22NM'] == lad]
    centroid_lat = wards['LAT'].mean()
    centroid_lon = wards['LONG'].mean()
    council_centroids[council] = (centroid_lat, centroid_lon)

# Verify
for council in council_compositions:
    total = sum(council_compositions[council].values())
    lat, lon = council_centroids[council]
    print(f'{council}: {total} seats | centroid: {lat:.4f}, {lon:.4f}')

Sheffield: 84 seats | centroid: 53.3816, -1.4765
Barnsley: 63 seats | centroid: 53.5493, -1.4585
Wakefield: 63 seats | centroid: 53.6749, -1.4231


In [39]:
import math

def make_pie_svg(composition, colours, size=80):
    total = sum(composition.values())
    svg_parts = []
    cx, cy, r = size/2, size/2, size/2 - 2
    
    start_angle = -math.pi / 2  # start from top
    
    for party, count in composition.items():
        if count == 0:
            continue
        slice_angle = 2 * math.pi * (count / total)
        end_angle = start_angle + slice_angle
        
        x1 = cx + r * math.cos(start_angle)
        y1 = cy + r * math.sin(start_angle)
        x2 = cx + r * math.cos(end_angle)
        y2 = cy + r * math.sin(end_angle)
        
        large_arc = 1 if slice_angle > math.pi else 0
        colour = colours.get(party, '#888888')
        
        path = f'M {cx},{cy} L {x1:.2f},{y1:.2f} A {r},{r} 0 {large_arc},1 {x2:.2f},{y2:.2f} Z'
        svg_parts.append(f'<path d="{path}" fill="{colour}" stroke="white" stroke-width="1.5"/>')
        
        start_angle = end_angle
    
    # Council label
    svg_parts.append(f'<circle cx="{cx}" cy="{cy}" r="{r*0.35}" fill="white"/>')
    
    svg = f'<svg width="{size}" height="{size}" xmlns="http://www.w3.org/2000/svg">{"".join(svg_parts)}</svg>'
    return svg

# Test it
test_svg = make_pie_svg(council_compositions['Sheffield'], party_colours)
print('SVG generated successfully')
print(f'Length: {len(test_svg)} chars')

SVG generated successfully
Length: 718 chars


In [10]:
# Special notes
special_notes = {
    'Firth Park': '⚠️ Two seats contested — Reform UK & Labour Party each won one seat.',
    'Wakefield East': '⚠️ Two seats contested — Labour Party & Reform UK each won one seat.'
}

# Base map
m = folium.Map(location=[53.6, -1.5], zoom_start=10, tiles='CartoDB positron')

# Inject CSS
meta = Element("""
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<style>
    .leaflet-tooltip {
        font-size: 14px !important;
        min-width: 260px !important;
        padding: 10px !important;
        pointer-events: none;
    }
</style>
""")
m.get_root().header.add_child(meta)

# FeatureGroups
councils = ['Sheffield', 'Barnsley', 'Wakefield']
council_groups = {}
for council in councils:
    council_groups[council] = folium.FeatureGroup(name=council, show=True)

grey_group = folium.FeatureGroup(name='No 2026 data', show=False)
pie_group = folium.FeatureGroup(name='Council composition', show=False)

# Ward polygons
for _, row in gdf_merged.iterrows():
    ward = row['Ward']
    lad = row['LAD22NM']
    party = row['Party']
    colour = row['colour']

    if pd.isna(party):
        target_group = grey_group
        tooltip_html = f'<div style="font-family:Arial; padding:8px;"><b>{ward}</b><br><span style="color:#888;">No 2026 election data</span></div>'
    else:
        council = df[df['Ward'] == ward]['Council'].iloc[0] if ward in df['Ward'].values else None
        target_group = council_groups.get(council, grey_group)

        results = ward_results.get((ward, lad), [])
        max_votes = max(r['Votes'] for r in results) if results else 1

        bars_html = ""
        for r in results:
            c = party_colours.get(r['Party'], '#888888')
            bar_width = int((r['Votes'] / max_votes) * 200)
            bars_html += f"""
            <div style="margin:6px 0; font-size:13px;">
                <div style="margin-bottom:2px;">{r['Party']}: <b>{int(r['Votes'])}</b></div>
                <div style="background:{c}; width:{bar_width}px; height:14px; border-radius:3px;"></div>
            </div>
            """

        note = special_notes.get(ward, '')
        note_html = f'<div style="font-size:11px; color:#888; margin-top:8px;">{note}</div>' if note else ''

        tooltip_html = f"""
        <div style="font-family:Arial; min-width:250px; padding:8px;">
            <b style="font-size:15px;">{ward}</b><br>
            <span style="font-size:12px; color:#888;">{council}</span><br><br>
            {bars_html}
            {note_html}
        </div>
        """

    folium.GeoJson(
        row['geometry'],
        style_function=lambda x, colour=colour: {
            'fillColor': colour,
            'color': 'white',
            'weight': 1.5,
            'fillOpacity': 0.5 if colour != '#cccccc' else 0.15
        },
        tooltip=folium.Tooltip(tooltip_html, sticky=True)
    ).add_to(target_group)

# Add council boundary outlines to pie group
for council, lad in [('Sheffield', 'Sheffield'), ('Barnsley', 'Barnsley'), ('Wakefield', 'Wakefield')]:
    council_wards = gdf[gdf['LAD22NM'] == lad]
    
    # Dissolve all wards into one council boundary
    council_boundary = council_wards.dissolve()
    
    folium.GeoJson(
        council_boundary,
        style_function=lambda x: {
            'fillColor': 'transparent',
            'color': '#333333',
            'weight': 2.5,
            'fillOpacity': 0,
            'dashArray': '5,5'
        },
        tooltip=folium.Tooltip(f'<b>{council}</b>', sticky=False)
    ).add_to(pie_group)

# Pie chart markers
for council, (lat, lon) in council_centroids.items():
    composition = council_compositions[council]
    total = sum(composition.values())
    svg = make_pie_svg(composition, party_colours, size=100)

    # Build tooltip for pie
    breakdown = ''.join([
        f'<div style="margin:4px 0; font-size:12px;">'
        f'<span style="background:{party_colours.get(p,"#888")}; width:10px; height:10px; '
        f'display:inline-block; border-radius:2px; margin-right:6px;"></span>'
        f'{p}: <b>{c}</b> seats</div>'
        for p, c in sorted(composition.items(), key=lambda x: -x[1])
    ])

    pie_tooltip = f"""
    <div style="font-family:Arial; min-width:200px; padding:8px;">
        <b style="font-size:15px;">{council}</b><br>
        <span style="font-size:12px; color:#888;">Full council composition</span><br><br>
        {breakdown}
        <div style="font-size:11px; color:#888; margin-top:6px;">Total: {total} seats</div>
    </div>
    """

    icon = folium.DivIcon(
        html=svg,
        icon_size=(100, 100),
        icon_anchor=(50, 50)
    )

    folium.Marker(
        location=[lat, lon],
        icon=icon,
        tooltip=folium.Tooltip(pie_tooltip, sticky=True)
    ).add_to(pie_group)

# Add all groups
for group in council_groups.values():
    group.add_to(m)
grey_group.add_to(m)
pie_group.add_to(m)

# Layer control
folium.LayerControl(collapsed=False).add_to(m)



# Legend
legend_html = """
<div style="position: fixed; bottom: 0; left: 0; right: 0; z-index: 1000; font-family: Arial;">
    <div style="text-align:center;">
        <button onclick="
            var panel = document.getElementById('legend-panel');
            var btn = document.getElementById('legend-toggle-btn');
            if(panel.style.display === 'none') {
                panel.style.display = 'block';
                btn.innerHTML = '▼ Hide legend';
            } else {
                panel.style.display = 'none';
                btn.innerHTML = '▲ Show legend';
            }"
            id="legend-toggle-btn"
            style="background:white; border:1px solid #ccc; border-radius:6px 6px 0 0;
                   padding:4px 16px; cursor:pointer; font-size:12px;
                   box-shadow: 0 -2px 6px rgba(0,0,0,0.15);">
            ▼ Hide legend
        </button>
    </div>
    <div id="legend-panel" style="background-color: white; padding: 8px 20px;
            box-shadow: 0 -2px 6px rgba(0,0,0,0.2); text-align: center;">
        <div style="white-space: nowrap; overflow-x: auto; font-size: 13px; margin-bottom: 4px;">
            {% for party, colour in parties.items() %}
            <span style="display:inline-block; margin: 0 10px;">
                <span style="background:{{ colour }}; width:12px; height:12px;
                             display:inline-block; border-radius:3px; margin-right:4px;
                             vertical-align:middle;"></span>
                {{ party }}
            </span>
            {% endfor %}
            <span style="display:inline-block; margin: 0 10px;">
                <span style="background:#cccccc; width:12px; height:12px;
                             display:inline-block; border-radius:3px; margin-right:4px;
                             vertical-align:middle;"></span>
                No 2026 data
            </span>
        </div>
        <div style="font-size:11px; color:#555; margin-bottom: 3px;">
            Source: Official results from Sheffield, Barnsley &amp; Wakefield Metropolitan Borough Councils
        </div>
        <div style="font-size:11px; color:#888;">
            Built by encephalon_zest · Co-authored with Claude (Anthropic) · 2026
        </div>
    </div>
</div>
"""

legend = Template(legend_html).render(parties=party_colours)
m.get_root().html.add_child(folium.Element(legend))

OUTPUT = '/home/martin/python_projects/Data_www/Yorkshire_map.html'
m.save(OUTPUT)
print('Done -- refresh your browser!')

Done -- refresh your browser!


In [1]:
# Find all split wards -- more than one elected party
split_wards = []

for (ward, lad), group in df.groupby(['Ward', 'LAD22NM']):
    elected = group[group['Elected'] == True]
    unique_parties = elected['Party'].unique()
    if len(unique_parties) > 1:
        split_wards.append({
            'Ward': ward,
            'LAD22NM': lad,
            'Parties': list(unique_parties)
        })

print(f'Split wards found: {len(split_wards)}')
for s in split_wards:
    print(f"  {s['Ward']} ({s['LAD22NM']}) -- {s['Parties']}")

NameError: name 'df' is not defined

In [2]:
import geopandas as gpd
import pandas as pd
import folium
import math
from jinja2 import Template
from branca.element import Element

# File paths
GEOJSON = '/home/martin/python_projects/Data_www/E12000003_clean.geojson'
CSV = '/home/martin/python_projects/Data_www/Yorkshire_unified.csv'

# Load both
gdf = gpd.read_file(GEOJSON)
df = pd.read_csv(CSV)

# Fix ward names
gdf['Ward'] = gdf['WD22NM'].str.strip()
df['Ward'] = df['Ward'].str.replace('\xa0', ' ', regex=False).str.strip()

# Add LAD22NM to df
df['LAD22NM'] = df['Council'].map({
    'Sheffield': 'Sheffield',
    'Barnsley': 'Barnsley',
    'Wakefield': 'Wakefield'
})

# Party colours
party_colours = {
    'Green Party':        '#4CAF50',
    'Liberal Democrats':  '#FAA61A',
    'Labour Party':       '#E4003B',
    'Reform UK':          '#12B6CF',
    'Independent':        '#888888',
    'Conservative Party': '#1D4F8C'
}

# Normalise party names
party_map = {
    'Labour and Co-operative Party': 'Labour Party',
    'ReformUK - Changing Politics for Good': 'Reform UK',
    'The Conservative Party Candidate': 'Conservative Party',
    'Conservative Party Candidate': 'Conservative Party',
    'Liberal Democrat Focus Team': 'Liberal Democrats'
}
df['Party'] = df['Party'].replace(party_map)

# Ward name fixes
ward_name_fixes = {
    'Athersley and New Lodge': 'St Helens',
    'Darfield and Great Houghton': 'Darfield',
    'Dodworth, Stainborough and Tankersley': 'Dodworth',
    'Knottingley and Ferrybridge': 'Knottingley'
}
df['Ward'] = df['Ward'].replace(ward_name_fixes)

# Winners
winners = df.loc[df.groupby(['Ward', 'LAD22NM'])['Votes'].idxmax()].reset_index(drop=True)

# Merge
gdf_merged = gdf.merge(winners[['Ward', 'LAD22NM', 'Party', 'Votes']],
                        on=['Ward', 'LAD22NM'],
                        how='left')
gdf_merged['colour'] = gdf_merged['Party'].map(party_colours).fillna('#cccccc')

# Ward results dict
ward_results = {}
for (ward, lad), group in df.groupby(['Ward', 'LAD22NM']):
    best_per_party = group.groupby('Party')['Votes'].max().reset_index()
    best_per_party = best_per_party.sort_values('Votes', ascending=False).head(3).reset_index(drop=True)
    ward_results[(ward, lad)] = best_per_party[['Party', 'Votes']].to_dict('records')

# Sheffield composition
sheffield_composition = {
    'Labour Party':      25,
    'Liberal Democrats': 22,
    'Green Party':       20,
    'Reform UK':         13,
    'Independent':        4
}

# Council compositions
council_compositions = {
    'Sheffield': sheffield_composition,
    'Barnsley': dict(df[(df['Council'] == 'Barnsley') & (df['Elected'] == True)].groupby('Party')['Party'].count()),
    'Wakefield': dict(df[(df['Council'] == 'Wakefield') & (df['Elected'] == True)].groupby('Party')['Party'].count())
}

# Council centroids
council_centroids = {}
for council, lad in [('Sheffield', 'Sheffield'), ('Barnsley', 'Barnsley'), ('Wakefield', 'Wakefield')]:
    wards = gdf[gdf['LAD22NM'] == lad]
    council_centroids[council] = (wards['LAT'].mean(), wards['LONG'].mean())

print('All reloaded cleanly!')
print(f'GeoJSON wards: {len(gdf)}')
print(f'CSV rows: {len(df)}')
print(f'Merged wards: {len(gdf_merged)}')

All reloaded cleanly!
GeoJSON wards: 411
CSV rows: 720
Merged wards: 411


In [3]:
# Find all split wards -- more than one elected party
split_wards = []

for (ward, lad), group in df.groupby(['Ward', 'LAD22NM']):
    elected = group[group['Elected'] == True]
    unique_parties = elected['Party'].unique()
    if len(unique_parties) > 1:
        split_wards.append({
            'Ward': ward,
            'LAD22NM': lad,
            'Parties': list(unique_parties)
        })

print(f'Split wards found: {len(split_wards)}')
for s in split_wards:
    print(f"  {s['Ward']} ({s['LAD22NM']}) -- {s['Parties']}")

Split wards found: 10
  Cudworth (Barnsley) -- ['Reform UK', 'Labour Party']
  Knottingley (Wakefield) -- ['Reform UK', 'Liberal Democrats']
  Penistone East (Barnsley) -- ['Labour Party', 'Reform UK']
  Penistone West (Barnsley) -- ['Liberal Democrats', 'Reform UK']
  Rockingham (Barnsley) -- ['Reform UK', 'Independent']
  Wakefield East (Wakefield) -- ['Labour Party', 'Reform UK']
  Wakefield North (Wakefield) -- ['Green Party', 'Reform UK']
  Wakefield South (Wakefield) -- ['Conservative Party', 'Reform UK']
  Wombwell (Barnsley) -- ['Reform UK', 'Labour Party']
  Worsbrough (Barnsley) -- ['Reform UK', 'Independent']


In [4]:
# Verify split wards -- show elected candidates per ward
for s in split_wards:
    ward = s['Ward']
    lad = s['LAD22NM']
    elected = df[(df['Ward'] == ward) & (df['LAD22NM'] == lad) & (df['Elected'] == True)]
    print(f"{ward} ({lad}) -- {len(elected)} seats elected:")
    for _, row in elected.iterrows():
        print(f"  {row['Party']} -- {row['Candidate']} -- {int(row['Votes'])} votes")
    print()

Cudworth (Barnsley) -- 3 seats elected:
  Reform UK -- Stanley John Bryan -- 1283 votes
  Labour Party -- Joe Hayward -- 1303 votes
  Labour Party -- Steve Houghton -- 1306 votes

Knottingley (Wakefield) -- 3 seats elected:
  Reform UK -- Garbutt, William -- 1502 votes
  Liberal Democrats -- Hayes, Adele -- 1430 votes
  Liberal Democrats -- Speak, Rachel -- 1425 votes

Penistone East (Barnsley) -- 3 seats elected:
  Labour Party -- Alex Burnett -- 2040 votes
  Reform UK -- Kay Hughes -- 1669 votes
  Labour Party -- Frances Nixon -- 1645 votes

Penistone West (Barnsley) -- 3 seats elected:
  Liberal Democrats -- Florentine Bootha-King -- 1562 votes
  Liberal Democrats -- David Greenhough -- 1680 votes
  Reform UK -- Vicky Jackson -- 1399 votes

Rockingham (Barnsley) -- 3 seats elected:
  Reform UK -- Christopher Denton -- 1291 votes
  Reform UK -- Pat Gregory -- 1462 votes
  Independent -- Andy Wray -- 1377 votes

Wakefield East (Wakefield) -- 3 seats elected:
  Labour Party -- Ayub, Yu

In [5]:
# Add Sheffield split ward manually
split_wards.append({
    'Ward': 'Firth Park',
    'LAD22NM': 'Sheffield',
    'Parties': ['Labour Party', 'Reform UK']
})

print(f'Total split wards: {len(split_wards)}')
for s in split_wards:
    print(f"  {s['Ward']} ({s['LAD22NM']}) -- {s['Parties']}")

Total split wards: 11
  Cudworth (Barnsley) -- ['Reform UK', 'Labour Party']
  Knottingley (Wakefield) -- ['Reform UK', 'Liberal Democrats']
  Penistone East (Barnsley) -- ['Labour Party', 'Reform UK']
  Penistone West (Barnsley) -- ['Liberal Democrats', 'Reform UK']
  Rockingham (Barnsley) -- ['Reform UK', 'Independent']
  Wakefield East (Wakefield) -- ['Labour Party', 'Reform UK']
  Wakefield North (Wakefield) -- ['Green Party', 'Reform UK']
  Wakefield South (Wakefield) -- ['Conservative Party', 'Reform UK']
  Wombwell (Barnsley) -- ['Reform UK', 'Labour Party']
  Worsbrough (Barnsley) -- ['Reform UK', 'Independent']
  Firth Park (Sheffield) -- ['Labour Party', 'Reform UK']


In [6]:
# Build stripe patterns for all split ward colour combos
def get_stripe_id(c1, c2):
    return f"stripe_{c1.replace('#','')}_{c2.replace('#','')}"

# Collect unique colour pairs
colour_pairs = set()
split_ward_lookup = {}
for s in split_wards:
    parties = s['Parties']
    c1 = party_colours.get(parties[0], '#888888')
    c2 = party_colours.get(parties[1], '#888888')
    colour_pairs.add((c1, c2))
    split_ward_lookup[(s['Ward'], s['LAD22NM'])] = (c1, c2)

# Generate SVG defs block
svg_defs = '<svg style="display:none"><defs>'
for c1, c2 in colour_pairs:
    pid = get_stripe_id(c1, c2)
    svg_defs += f"""
    <pattern id="{pid}" patternUnits="userSpaceOnUse" width="10" height="10" patternTransform="rotate(45)">
        <rect width="5" height="10" fill="{c1}"/>
        <rect x="5" width="5" height="10" fill="{c2}"/>
    </pattern>
    """
svg_defs += '</defs></svg>'

print('Stripe patterns generated:')
for c1, c2 in colour_pairs:
    print(f'  {get_stripe_id(c1, c2)}')
print()
print(f'Split ward lookup: {len(split_ward_lookup)} wards')

Stripe patterns generated:
  stripe_12B6CF_888888
  stripe_4CAF50_12B6CF
  stripe_1D4F8C_12B6CF
  stripe_E4003B_12B6CF
  stripe_12B6CF_E4003B
  stripe_12B6CF_FAA61A
  stripe_FAA61A_12B6CF

Split ward lookup: 11 wards


In [9]:
# Special notes
special_notes = {
    'Firth Park': '⚠️ Two seats contested — Labour Party & Reform UK each won one seat.',
    'Wakefield East': '⚠️ Two seats contested — Labour Party & Reform UK each won one seat.',
    'Cudworth': '⚠️ Three seats — Reform UK 1, Labour Party 2.',
    'Penistone East': '⚠️ Three seats — Labour Party 2, Reform UK 1.',
    'Penistone West': '⚠️ Three seats — Liberal Democrats 2, Reform UK 1.',
    'Rockingham': '⚠️ Three seats — Reform UK 2, Independent 1.',
    'Knottingley': '⚠️ Three seats — Reform UK 1, Liberal Democrats 2.',
    'Wakefield North': '⚠️ Three seats — Green Party 1, Reform UK 2.',
    'Wakefield South': '⚠️ Three seats — Conservative Party 1, Reform UK 2.',
    'Wombwell': '⚠️ Three seats — Reform UK 2, Labour Party 1.',
    'Worsbrough': '⚠️ Three seats — Reform UK 2, Independent 1.'
}

# Base map
m = folium.Map(location=[53.6, -1.5], zoom_start=10, tiles='CartoDB positron')

# Inject CSS + SVG stripe patterns
meta = Element(f"""
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<style>
    .leaflet-tooltip {{
        font-size: 14px !important;
        min-width: 260px !important;
        padding: 10px !important;
        pointer-events: none;
    }}
</style>
{svg_defs}
""")
m.get_root().header.add_child(meta)

# FeatureGroups
councils = ['Sheffield', 'Barnsley', 'Wakefield']
council_groups = {}
for council in councils:
    council_groups[council] = folium.FeatureGroup(name=council, show=True)

grey_group = folium.FeatureGroup(name='No 2026 data', show=False)
pie_group = folium.FeatureGroup(name='Council composition', show=False)

# Ward polygons
for _, row in gdf_merged.iterrows():
    ward = row['Ward']
    lad = row['LAD22NM']
    party = row['Party']
    colour = row['colour']

    if pd.isna(party):
        target_group = grey_group
        tooltip_html = f'<div style="font-family:Arial; padding:8px;"><b>{ward}</b><br><span style="color:#888;">No 2026 election data</span></div>'
        fill_colour = '#cccccc'
        fill_opacity = 0.15
        pattern_url = None
    else:
        council = df[df['Ward'] == ward]['Council'].iloc[0] if ward in df['Ward'].values else None
        target_group = council_groups.get(council, grey_group)

        results = ward_results.get((ward, lad), [])
        max_votes = max(r['Votes'] for r in results) if results else 1

        bars_html = ""
        for r in results:
            c = party_colours.get(r['Party'], '#888888')
            bar_width = int((r['Votes'] / max_votes) * 200)
            bars_html += f"""
            <div style="margin:6px 0; font-size:13px;">
                <div style="margin-bottom:2px;">{r['Party']}: <b>{int(r['Votes'])}</b></div>
                <div style="background:{c}; width:{bar_width}px; height:14px; border-radius:3px;"></div>
            </div>
            """

        note = special_notes.get(ward, '')
        note_html = f'<div style="font-size:11px; color:#888; margin-top:8px;">{note}</div>' if note else ''

        tooltip_html = f"""
        <div style="font-family:Arial; min-width:250px; padding:8px;">
            <b style="font-size:15px;">{ward}</b><br>
            <span style="font-size:12px; color:#888;">{council}</span><br><br>
            {bars_html}
            {note_html}
        </div>
        """

        # Check if split ward
        if (ward, lad) in split_ward_lookup:
            c1, c2 = split_ward_lookup[(ward, lad)]
            pid = get_stripe_id(c1, c2)
            fill_colour = f'url(#{pid})'
            fill_opacity = 0.8
        else:
            fill_colour = colour
            fill_opacity = 0.5

    folium.GeoJson(
        row['geometry'],
        style_function=lambda x, fc=fill_colour, fo=fill_opacity: {
            'fillColor': fc,
            'color': 'white',
            'weight': 1.5,
            'fillOpacity': fo
        },
        tooltip=folium.Tooltip(tooltip_html, sticky=True)
    ).add_to(target_group)

# Add council boundary outlines and pie charts to pie group
for council, lad in [('Sheffield', 'Sheffield'), ('Barnsley', 'Barnsley'), ('Wakefield', 'Wakefield')]:
    council_wards = gdf[gdf['LAD22NM'] == lad]
    council_boundary = council_wards.dissolve()

    folium.GeoJson(
        council_boundary,
        style_function=lambda x: {
            'fillColor': 'transparent',
            'color': '#333333',
            'weight': 2.5,
            'fillOpacity': 0,
            'dashArray': '5,5'
        },
        tooltip=folium.Tooltip(f'<b>{council}</b>', sticky=False)
    ).add_to(pie_group)

    # Pie chart
    composition = council_compositions[council]
    total = sum(composition.values())
    svg = make_pie_svg(composition, party_colours, size=100)

    breakdown = ''.join([
        f'<div style="margin:4px 0; font-size:12px;">'
        f'<span style="background:{party_colours.get(p,"#888")}; width:10px; height:10px; '
        f'display:inline-block; border-radius:2px; margin-right:6px;"></span>'
        f'{p}: <b>{c}</b> seats</div>'
        for p, c in sorted(composition.items(), key=lambda x: -x[1])
    ])

    pie_tooltip = f"""
    <div style="font-family:Arial; min-width:200px; padding:8px;">
        <b style="font-size:15px;">{council}</b><br>
        <span style="font-size:12px; color:#888;">Full council composition</span><br><br>
        {breakdown}
        <div style="font-size:11px; color:#888; margin-top:6px;">Total: {total} seats</div>
    </div>
    """

    icon = folium.DivIcon(
        html=svg,
        icon_size=(100, 100),
        icon_anchor=(50, 50)
    )

    lat, lon = council_centroids[council]
    folium.Marker(
        location=[lat, lon],
        icon=icon,
        tooltip=folium.Tooltip(pie_tooltip, sticky=True)
    ).add_to(pie_group)

# Add all groups
for group in council_groups.values():
    group.add_to(m)
grey_group.add_to(m)
pie_group.add_to(m)

# Layer control
folium.LayerControl(collapsed=False).add_to(m)

# Legend
legend_html = """
<div style="position: fixed; bottom: 0; left: 0; right: 0; z-index: 1000; font-family: Arial;">
    <div style="text-align:center;">
        <button onclick="
            var panel = document.getElementById('legend-panel');
            var btn = document.getElementById('legend-toggle-btn');
            if(panel.style.display === 'none') {
                panel.style.display = 'block';
                btn.innerHTML = '▼ Hide legend';
            } else {
                panel.style.display = 'none';
                btn.innerHTML = '▲ Show legend';
            }"
            id="legend-toggle-btn"
            style="background:white; border:1px solid #ccc; border-radius:6px 6px 0 0;
                   padding:4px 16px; cursor:pointer; font-size:12px;
                   box-shadow: 0 -2px 6px rgba(0,0,0,0.15);">
            ▼ Hide legend
        </button>
    </div>
    <div id="legend-panel" style="background-color: white; padding: 8px 20px;
            box-shadow: 0 -2px 6px rgba(0,0,0,0.2); text-align: center;">
        <div style="white-space: nowrap; overflow-x: auto; font-size: 13px; margin-bottom: 4px;">
            {% for party, colour in parties.items() %}
            <span style="display:inline-block; margin: 0 10px;">
                <span style="background:{{ colour }}; width:12px; height:12px;
                             display:inline-block; border-radius:3px; margin-right:4px;
                             vertical-align:middle;"></span>
                {{ party }}
            </span>
            {% endfor %}
            <span style="display:inline-block; margin: 0 10px;">
                <span style="background:#cccccc; width:12px; height:12px;
                             display:inline-block; border-radius:3px; margin-right:4px;
                             vertical-align:middle;"></span>
                No 2026 data
            </span>
            <span style="display:inline-block; margin: 0 10px;">
                <span style="background: repeating-linear-gradient(45deg, #12B6CF, #12B6CF 5px, #E4003B 5px, #E4003B 10px);
                             width:12px; height:12px;
                             display:inline-block; border-radius:3px; margin-right:4px;
                             vertical-align:middle;"></span>
                Split result
            </span>
        </div>
        <div style="font-size:11px; color:#555; margin-bottom: 3px;">
            Source: Official results from Sheffield, Barnsley &amp; Wakefield Metropolitan Borough Councils
        </div>
        <div style="font-size:11px; color:#888;">
            Built by encephalon_zest · Co-authored with Claude (Anthropic) · 2026
        </div>
    </div>
</div>
"""

legend = Template(legend_html).render(parties=party_colours)
m.get_root().html.add_child(folium.Element(legend))

OUTPUT = '/home/martin/python_projects/Data_www/Yorkshire_map.html'
m.save(OUTPUT)
print('Done -- refresh your browser!')

Done -- refresh your browser!


In [8]:
import math

def make_pie_svg(composition, colours, size=80):
    total = sum(composition.values())
    svg_parts = []
    cx, cy, r = size/2, size/2, size/2 - 2
    start_angle = -math.pi / 2

    for party, count in composition.items():
        if count == 0:
            continue
        slice_angle = 2 * math.pi * (count / total)
        end_angle = start_angle + slice_angle

        x1 = cx + r * math.cos(start_angle)
        y1 = cy + r * math.sin(start_angle)
        x2 = cx + r * math.cos(end_angle)
        y2 = cy + r * math.sin(end_angle)

        large_arc = 1 if slice_angle > math.pi else 0
        colour = colours.get(party, '#888888')

        path = f'M {cx},{cy} L {x1:.2f},{y1:.2f} A {r},{r} 0 {large_arc},1 {x2:.2f},{y2:.2f} Z'
        svg_parts.append(f'<path d="{path}" fill="{colour}" stroke="white" stroke-width="1.5"/>')
        start_angle = end_angle

    svg_parts.append(f'<circle cx="{cx}" cy="{cy}" r="{r*0.35}" fill="white"/>')
    svg = f'<svg width="{size}" height="{size}" xmlns="http://www.w3.org/2000/svg">{"".join(svg_parts)}</svg>'
    return svg

print('make_pie_svg ready!')

make_pie_svg ready!


In [11]:
# Special notes
special_notes = {
    'Firth Park': '⚠️ Two seats contested — Labour Party & Reform UK each won one seat.',
    'Wakefield East': '⚠️ Three seats — Labour Party 1, Reform UK 2.',
    'Cudworth': '⚠️ Three seats — Reform UK 1, Labour Party 2.',
    'Penistone East': '⚠️ Three seats — Labour Party 2, Reform UK 1.',
    'Penistone West': '⚠️ Three seats — Liberal Democrats 2, Reform UK 1.',
    'Rockingham': '⚠️ Three seats — Reform UK 2, Independent 1.',
    'Knottingley': '⚠️ Three seats — Reform UK 1, Liberal Democrats 2.',
    'Wakefield North': '⚠️ Three seats — Green Party 1, Reform UK 2.',
    'Wakefield South': '⚠️ Three seats — Conservative Party 1, Reform UK 2.',
    'Wombwell': '⚠️ Three seats — Reform UK 2, Labour Party 1.',
    'Worsbrough': '⚠️ Three seats — Reform UK 2, Independent 1.'
}

# Base map
m = folium.Map(location=[53.6, -1.5], zoom_start=10, tiles='CartoDB positron')

# Inject CSS
meta = Element("""
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<style>
    .leaflet-tooltip {
        font-size: 14px !important;
        min-width: 260px !important;
        padding: 10px !important;
        pointer-events: none;
    }
</style>
""")
m.get_root().header.add_child(meta)

# FeatureGroups
councils = ['Sheffield', 'Barnsley', 'Wakefield']
council_groups = {}
for council in councils:
    council_groups[council] = folium.FeatureGroup(name=council, show=True)

grey_group = folium.FeatureGroup(name='No 2026 data', show=False)
pie_group = folium.FeatureGroup(name='Council composition', show=False)

# Ward polygons
for _, row in gdf_merged.iterrows():
    ward = row['Ward']
    lad = row['LAD22NM']
    party = row['Party']
    colour = row['colour']

    if pd.isna(party):
        target_group = grey_group
        tooltip_html = f'<div style="font-family:Arial; padding:8px;"><b>{ward}</b><br><span style="color:#888;">No 2026 election data</span></div>'
        fill_colour = '#cccccc'
        fill_opacity = 0.15
    else:
        council = df[df['Ward'] == ward]['Council'].iloc[0] if ward in df['Ward'].values else None
        target_group = council_groups.get(council, grey_group)

        results = ward_results.get((ward, lad), [])
        max_votes = max(r['Votes'] for r in results) if results else 1

        bars_html = ""
        for r in results:
            c = party_colours.get(r['Party'], '#888888')
            bar_width = int((r['Votes'] / max_votes) * 200)
            bars_html += f"""
            <div style="margin:6px 0; font-size:13px;">
                <div style="margin-bottom:2px;">{r['Party']}: <b>{int(r['Votes'])}</b></div>
                <div style="background:{c}; width:{bar_width}px; height:14px; border-radius:3px;"></div>
            </div>
            """

        note = special_notes.get(ward, '')
        note_html = f'<div style="font-size:11px; color:#888; margin-top:8px;">{note}</div>' if note else ''

        tooltip_html = f"""
        <div style="font-family:Arial; min-width:250px; padding:8px;">
            <b style="font-size:15px;">{ward}</b><br>
            <span style="font-size:12px; color:#888;">{council}</span><br><br>
            {bars_html}
            {note_html}
        </div>
        """
        fill_colour = colour
        fill_opacity = 0.5

    folium.GeoJson(
        row['geometry'],
        style_function=lambda x, fc=fill_colour, fo=fill_opacity: {
            'fillColor': fc,
            'color': 'white',
            'weight': 1.5,
            'fillOpacity': fo
        },
        tooltip=folium.Tooltip(tooltip_html, sticky=True)
    ).add_to(target_group)

# Council boundaries and pie charts
for council, lad in [('Sheffield', 'Sheffield'), ('Barnsley', 'Barnsley'), ('Wakefield', 'Wakefield')]:
    council_wards = gdf[gdf['LAD22NM'] == lad]
    council_boundary = council_wards.dissolve()

    folium.GeoJson(
        council_boundary,
        style_function=lambda x: {
            'fillColor': 'transparent',
            'color': '#333333',
            'weight': 2.5,
            'fillOpacity': 0,
            'dashArray': '5,5'
        },
        tooltip=folium.Tooltip(f'<b>{council}</b>', sticky=False)
    ).add_to(pie_group)

    composition = council_compositions[council]
    total = sum(composition.values())
    svg = make_pie_svg(composition, party_colours, size=100)

    breakdown = ''.join([
        f'<div style="margin:4px 0; font-size:12px;">'
        f'<span style="background:{party_colours.get(p,"#888")}; width:10px; height:10px; '
        f'display:inline-block; border-radius:2px; margin-right:6px;"></span>'
        f'{p}: <b>{c}</b> seats</div>'
        for p, c in sorted(composition.items(), key=lambda x: -x[1])
    ])

    pie_tooltip = f"""
    <div style="font-family:Arial; min-width:200px; padding:8px;">
        <b style="font-size:15px;">{council}</b><br>
        <span style="font-size:12px; color:#888;">Full council composition</span><br><br>
        {breakdown}
        <div style="font-size:11px; color:#888; margin-top:6px;">Total: {total} seats</div>
    </div>
    """

    icon = folium.DivIcon(
        html=svg,
        icon_size=(100, 100),
        icon_anchor=(50, 50)
    )

    lat, lon = council_centroids[council]
    folium.Marker(
        location=[lat, lon],
        icon=icon,
        tooltip=folium.Tooltip(pie_tooltip, sticky=True)
    ).add_to(pie_group)

# Add all groups
for group in council_groups.values():
    group.add_to(m)
grey_group.add_to(m)
pie_group.add_to(m)

# Layer control
folium.LayerControl(collapsed=False).add_to(m)

# JS opacity toggle
js_opacity = """
<script>
document.addEventListener('DOMContentLoaded', function() {
    setTimeout(function() {
        var map = Object.values(window).find(function(v) {
            return v && v._leaflet_id && v.eachLayer;
        });
        if (!map) return;

        map.on('overlayadd overlayremove', function(e) {
            if (e.name === 'Council composition') {
                var dimmed = (e.type === 'overlayadd');
                map.eachLayer(function(layer) {
                    if (layer.eachLayer) {
                        layer.eachLayer(function(sublayer) {
                            if (sublayer.setStyle) {
                                sublayer.setStyle({
                                    fillOpacity: dimmed ? 0.12 : 0.5
                                });
                            }
                        });
                    }
                });
            }
        });
    }, 1000);
});
</script>
"""

m.get_root().html.add_child(folium.Element(js_opacity))

# Legend
legend_html = """
<div style="position: fixed; bottom: 0; left: 0; right: 0; z-index: 1000; font-family: Arial;">
    <div style="text-align:center;">
        <button onclick="
            var panel = document.getElementById('legend-panel');
            var btn = document.getElementById('legend-toggle-btn');
            if(panel.style.display === 'none') {
                panel.style.display = 'block';
                btn.innerHTML = '▼ Hide legend';
            } else {
                panel.style.display = 'none';
                btn.innerHTML = '▲ Show legend';
            }"
            id="legend-toggle-btn"
            style="background:white; border:1px solid #ccc; border-radius:6px 6px 0 0;
                   padding:4px 16px; cursor:pointer; font-size:12px;
                   box-shadow: 0 -2px 6px rgba(0,0,0,0.15);">
            ▼ Hide legend
        </button>
    </div>
    <div id="legend-panel" style="background-color: white; padding: 8px 20px;
            box-shadow: 0 -2px 6px rgba(0,0,0,0.2); text-align: center;">
        <div style="white-space: nowrap; overflow-x: auto; font-size: 13px; margin-bottom: 4px;">
            {% for party, colour in parties.items() %}
            <span style="display:inline-block; margin: 0 10px;">
                <span style="background:{{ colour }}; width:12px; height:12px;
                             display:inline-block; border-radius:3px; margin-right:4px;
                             vertical-align:middle;"></span>
                {{ party }}
            </span>
            {% endfor %}
            <span style="display:inline-block; margin: 0 10px;">
                <span style="background:#cccccc; width:12px; height:12px;
                             display:inline-block; border-radius:3px; margin-right:4px;
                             vertical-align:middle;"></span>
                No 2026 data
            </span>
        </div>
        <div style="font-size:11px; color:#555; margin-bottom: 3px;">
            Source: Official results from Sheffield, Barnsley &amp; Wakefield Metropolitan Borough Councils
        </div>
        <div style="font-size:11px; color:#888;">
            Built by encephalon_zest · Co-authored with Claude (Anthropic) · 2026
        </div>
    </div>
</div>
"""

legend = Template(legend_html).render(parties=party_colours)
m.get_root().html.add_child(folium.Element(legend))

OUTPUT = '/home/martin/python_projects/Data_www/Yorkshire_map.html'
m.save(OUTPUT)
print('Done -- refresh and test Council composition toggle!')

Done -- refresh and test Council composition toggle!


In [12]:
js_opacity = """
<script>
    setTimeout(function() {
        var checkboxes = document.querySelectorAll('.leaflet-control-layers-overlays input[type=checkbox]');
        var compositionCheckbox = null;
        
        checkboxes.forEach(function(cb) {
            var label = cb.closest('label');
            if (label && label.textContent.trim() === 'Council composition') {
                compositionCheckbox = cb;
            }
        });

        if (!compositionCheckbox) return;

        compositionCheckbox.addEventListener('change', function() {
            var dimmed = this.checked;
            var paths = document.querySelectorAll('.leaflet-overlay-pane svg path');
            paths.forEach(function(path) {
                var currentOpacity = parseFloat(path.style.fillOpacity || path.getAttribute('fill-opacity') || 0.5);
                if (dimmed) {
                    path.setAttribute('data-original-opacity', currentOpacity);
                    path.style.fillOpacity = 0.12;
                } else {
                    var original = path.getAttribute('data-original-opacity') || 0.5;
                    path.style.fillOpacity = original;
                }
            });
        });
    }, 2000);
</script>
"""

m.get_root().html.add_child(folium.Element(js_opacity))

OUTPUT = '/home/martin/python_projects/Data_www/Yorkshire_map.html'
m.save(OUTPUT)
print('Done -- refresh and test!')

Done -- refresh and test!


In [13]:
js_opacity = """
<script>
    setTimeout(function() {
        var checkboxes = document.querySelectorAll('.leaflet-control-layers-overlays input[type=checkbox]');
        var compositionCheckbox = null;
        
        checkboxes.forEach(function(cb) {
            var label = cb.closest('label');
            if (label && label.textContent.trim() === 'Council composition') {
                compositionCheckbox = cb;
            }
        });

        if (!compositionCheckbox) return;

        compositionCheckbox.addEventListener('change', function() {
            var dimmed = this.checked;
            var paths = document.querySelectorAll('.leaflet-overlay-pane svg path');
            paths.forEach(function(path) {
                var currentOpacity = parseFloat(path.style.fillOpacity || path.getAttribute('fill-opacity') || 0.5);
                if (dimmed) {
                    path.setAttribute('data-original-opacity', currentOpacity);
                    path.style.fillOpacity = 0.25;
                } else {
                    var original = path.getAttribute('data-original-opacity') || 0.5;
                    path.style.fillOpacity = original;
                }
            });
        });
    }, 2000);
</script>
"""

m.get_root().html.add_child(folium.Element(js_opacity))

OUTPUT = '/home/martin/python_projects/Data_www/Yorkshire_map.html'
m.save(OUTPUT)
print('Done -- refresh and test!')

Done -- refresh and test!


In [14]:
js_opacity = """
<script>
    setTimeout(function() {
        var checkboxes = document.querySelectorAll('.leaflet-control-layers-overlays input[type=checkbox]');
        var compositionCheckbox = null;
        
        checkboxes.forEach(function(cb) {
            var label = cb.closest('label');
            if (label && label.textContent.trim() === 'Council composition') {
                compositionCheckbox = cb;
            }
        });

        if (!compositionCheckbox) return;

        compositionCheckbox.addEventListener('change', function() {
            var dimmed = this.checked;
            var paths = document.querySelectorAll('.leaflet-overlay-pane svg path');
            
            paths.forEach(function(path) {
                if (dimmed) {
                    // Store original before dimming
                    if (!path.getAttribute('data-original-opacity')) {
                        var current = path.style.fillOpacity || 
                                     path.getAttribute('fill-opacity') || '0.5';
                        path.setAttribute('data-original-opacity', current);
                    }
                    path.style.fillOpacity = 0.25;
                } else {
                    // Restore original
                    var original = path.getAttribute('data-original-opacity');
                    if (original) {
                        path.style.fillOpacity = original;
                    } else {
                        path.style.fillOpacity = 0.5;
                    }
                }
            });
        });
    }, 2000);
</script>
"""

m.get_root().html.add_child(folium.Element(js_opacity))

OUTPUT = '/home/martin/python_projects/Data_www/Yorkshire_map.html'
m.save(OUTPUT)
print('Done -- refresh and test both directions!')

Done -- refresh and test both directions!


In [15]:
js_opacity = """
<script>
    setTimeout(function() {
        var checkboxes = document.querySelectorAll('.leaflet-control-layers-overlays input[type=checkbox]');
        var compositionCheckbox = null;
        
        checkboxes.forEach(function(cb) {
            var label = cb.closest('label');
            if (label && label.textContent.trim() === 'Council composition') {
                compositionCheckbox = cb;
            }
        });

        if (!compositionCheckbox) return;

        compositionCheckbox.addEventListener('change', function() {
            var dimmed = this.checked;
            var paths = document.querySelectorAll('.leaflet-overlay-pane svg path');
            paths.forEach(function(path) {
                path.style.fillOpacity = dimmed ? 0.25 : 0.5;
            });
        });
    }, 2000);
</script>
"""

m.get_root().html.add_child(folium.Element(js_opacity))

OUTPUT = '/home/martin/python_projects/Data_www/Yorkshire_map.html'
m.save(OUTPUT)
print('Done -- refresh and test!')

Done -- refresh and test!


In [16]:
js_opacity = """
<script>
    setTimeout(function() {
        var checkboxes = document.querySelectorAll('.leaflet-control-layers-overlays input[type=checkbox]');
        var compositionCheckbox = null;
        
        checkboxes.forEach(function(cb) {
            var label = cb.closest('label');
            if (label && label.textContent.trim() === 'Council composition') {
                compositionCheckbox = cb;
            }
        });

        if (!compositionCheckbox) return;

        // CC toggle
        compositionCheckbox.addEventListener('change', function() {
            var dimmed = this.checked;
            var paths = document.querySelectorAll('.leaflet-overlay-pane svg path');
            paths.forEach(function(path) {
                path.style.fillOpacity = dimmed ? 0.25 : 0.5;
            });
        });

        // Watch all OTHER checkboxes -- ward layer toggles
        checkboxes.forEach(function(cb) {
            if (cb === compositionCheckbox) return;
            cb.addEventListener('change', function() {
                // If CC is active, re-enforce dim after short delay
                if (compositionCheckbox.checked) {
                    setTimeout(function() {
                        var paths = document.querySelectorAll('.leaflet-overlay-pane svg path');
                        paths.forEach(function(path) {
                            path.style.fillOpacity = 0.25;
                        });
                    }, 100);
                }
            });
        });

    }, 2000);
</script>
"""

m.get_root().html.add_child(folium.Element(js_opacity))

OUTPUT = '/home/martin/python_projects/Data_www/Yorkshire_map.html'
m.save(OUTPUT)
print('Done -- refresh and test!')

Done -- refresh and test!


In [17]:
js_opacity = """
<script>
    setTimeout(function() {
        var checkboxes = document.querySelectorAll('.leaflet-control-layers-overlays input[type=checkbox]');
        var compositionCheckbox = null;
        
        checkboxes.forEach(function(cb) {
            var label = cb.closest('label');
            if (label && label.textContent.trim() === 'Council composition') {
                compositionCheckbox = cb;
            }
        });

        if (!compositionCheckbox) return;

        function applyTransitions(paths) {
            paths.forEach(function(path) {
                path.style.transition = 'fill-opacity 0.6s ease';
            });
        }

        // CC toggle
        compositionCheckbox.addEventListener('change', function() {
            var dimmed = this.checked;
            var paths = document.querySelectorAll('.leaflet-overlay-pane svg path');
            applyTransitions(paths);
            paths.forEach(function(path) {
                path.style.fillOpacity = dimmed ? 0.25 : 0.5;
            });
        });

        // Ward layer toggles
        checkboxes.forEach(function(cb) {
            if (cb === compositionCheckbox) return;
            cb.addEventListener('change', function() {
                if (compositionCheckbox.checked) {
                    setTimeout(function() {
                        var paths = document.querySelectorAll('.leaflet-overlay-pane svg path');
                        applyTransitions(paths);
                        paths.forEach(function(path) {
                            path.style.fillOpacity = 0.25;
                        });
                    }, 300);
                }
            });
        });

    }, 2000);
</script>
"""

m.get_root().html.add_child(folium.Element(js_opacity))

OUTPUT = '/home/martin/python_projects/Data_www/Yorkshire_map.html'
m.save(OUTPUT)
print('Done -- refresh and test!')

Done -- refresh and test!


In [18]:
js_opacity = """
<script>
    setTimeout(function() {
        var checkboxes = document.querySelectorAll('.leaflet-control-layers-overlays input[type=checkbox]');
        var compositionCheckbox = null;
        
        checkboxes.forEach(function(cb) {
            var label = cb.closest('label');
            if (label && label.textContent.trim() === 'Council composition') {
                compositionCheckbox = cb;
            }
        });

        if (!compositionCheckbox) return;

        function applyDim() {
            var paths = document.querySelectorAll('.leaflet-overlay-pane svg path');
            paths.forEach(function(path) {
                path.style.transition = 'fill-opacity 0.6s ease';
                path.style.fillOpacity = 0.25;
            });
        }

        function applyFull() {
            var paths = document.querySelectorAll('.leaflet-overlay-pane svg path');
            paths.forEach(function(path) {
                path.style.transition = 'fill-opacity 0.6s ease';
                path.style.fillOpacity = 0.5;
            });
        }

        // CC toggle
        compositionCheckbox.addEventListener('change', function() {
            if (this.checked) {
                applyDim();
            } else {
                applyFull();
            }
        });

        // Ward layer toggles -- use MutationObserver to catch Leaflet redraws
        var observer = new MutationObserver(function() {
            if (compositionCheckbox.checked) {
                // Small delay to let Leaflet finish redrawing
                setTimeout(applyDim, 50);
            }
        });

        var pane = document.querySelector('.leaflet-overlay-pane');
        if (pane) {
            observer.observe(pane, { childList: true, subtree: true });
        }

    }, 2000);
</script>
"""

m.get_root().html.add_child(folium.Element(js_opacity))

OUTPUT = '/home/martin/python_projects/Data_www/Yorkshire_map.html'
m.save(OUTPUT)
print('Done -- refresh and test!')

Done -- refresh and test!


In [19]:
# Update boundary style_function to add a class
folium.GeoJson(
    council_boundary,
    style_function=lambda x: {
        'fillColor': 'transparent',
        'color': '#999999',
        'weight': 3,
        'fillOpacity': 0,
        'dashArray': '8,6',
        'opacity': 0.4
    },
    tooltip=folium.Tooltip(f'<b>{council}</b>', sticky=False),
    name='boundary'
).add_to(pie_group)

In [20]:
js_opacity = """
<script>
    setTimeout(function() {
        var checkboxes = document.querySelectorAll('.leaflet-control-layers-overlays input[type=checkbox]');
        var compositionCheckbox = null;
        
        checkboxes.forEach(function(cb) {
            var label = cb.closest('label');
            if (label && label.textContent.trim() === 'Council composition') {
                compositionCheckbox = cb;
            }
        });

        if (!compositionCheckbox) return;

        function isWardPath(path) {
            // Skip boundary lines -- they have no fill or transparent fill
            var fill = path.getAttribute('fill');
            var fillOpacity = parseFloat(path.getAttribute('fill-opacity') || 1);
            if (fill === 'none' || fill === 'transparent' || fillOpacity === 0) return false;
            return true;
        }

        function applyDim() {
            var paths = document.querySelectorAll('.leaflet-overlay-pane svg path');
            paths.forEach(function(path) {
                if (!isWardPath(path)) return;
                path.style.transition = 'fill-opacity 0.6s ease';
                path.style.fillOpacity = 0.25;
            });
        }

        function applyFull() {
            var paths = document.querySelectorAll('.leaflet-overlay-pane svg path');
            paths.forEach(function(path) {
                if (!isWardPath(path)) return;
                path.style.transition = 'fill-opacity 0.6s ease';
                path.style.fillOpacity = 0.5;
            });
        }

        // CC toggle
        compositionCheckbox.addEventListener('change', function() {
            if (this.checked) {
                applyDim();
            } else {
                applyFull();
            }
        });

        // MutationObserver for ward layer toggles
        var observer = new MutationObserver(function() {
            if (compositionCheckbox.checked) {
                setTimeout(applyDim, 50);
            }
        });

        var pane = document.querySelector('.leaflet-overlay-pane');
        if (pane) {
            observer.observe(pane, { childList: true, subtree: true });
        }

    }, 2000);
</script>
"""

m.get_root().html.add_child(folium.Element(js_opacity))

OUTPUT = '/home/martin/python_projects/Data_www/Yorkshire_map.html'
m.save(OUTPUT)
print('Done -- refresh and test!')

Done -- refresh and test!


In [21]:
# Clean council boundaries -- separate from ward polygons
for council, lad in [('Sheffield', 'Sheffield'), ('Barnsley', 'Barnsley'), ('Wakefield', 'Wakefield')]:
    council_wards = gdf[gdf['LAD22NM'] == lad]
    council_boundary = council_wards.dissolve()

    folium.GeoJson(
        council_boundary,
        style_function=lambda x: {
            'fillColor': 'transparent',
            'color': '#444444',
            'weight': 2,
            'fillOpacity': 0,
            'opacity': 0.6
        }
    ).add_to(pie_group)

In [33]:
# Special notes
special_notes = {
    'Firth Park': '⚠️ Two seats contested — Labour Party & Reform UK each won one seat.',
    'Wakefield East': '⚠️ Three seats — Labour Party 1, Reform UK 2.',
    'Cudworth': '⚠️ Three seats — Reform UK 1, Labour Party 2.',
    'Penistone East': '⚠️ Three seats — Labour Party 2, Reform UK 1.',
    'Penistone West': '⚠️ Three seats — Liberal Democrats 2, Reform UK 1.',
    'Rockingham': '⚠️ Three seats — Reform UK 2, Independent 1.',
    'Knottingley': '⚠️ Three seats — Reform UK 1, Liberal Democrats 2.',
    'Wakefield North': '⚠️ Three seats — Green Party 1, Reform UK 2.',
    'Wakefield South': '⚠️ Three seats — Conservative Party 1, Reform UK 2.',
    'Wombwell': '⚠️ Three seats — Reform UK 2, Labour Party 1.',
    'Worsbrough': '⚠️ Three seats — Reform UK 2, Independent 1.'
}

# Base map
m = folium.Map(location=[53.6, -1.5], zoom_start=10, tiles='CartoDB positron')

# Inject CSS
meta = Element("""
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<style>
    .leaflet-tooltip {
        font-size: 14px !important;
        min-width: 260px !important;
        padding: 10px !important;
        pointer-events: none;
    }
</style>
""")
m.get_root().header.add_child(meta)

# FeatureGroups
councils = ['Sheffield', 'Barnsley', 'Wakefield']
council_groups = {}
for council in councils:
    council_groups[council] = folium.FeatureGroup(name=council, show=True)

grey_group = folium.FeatureGroup(name='No 2026 data', show=False)
pie_group = folium.FeatureGroup(name='Council composition', show=False)

# Ward polygons
for _, row in gdf_merged.iterrows():
    ward = row['Ward']
    lad = row['LAD22NM']
    party = row['Party']
    colour = row['colour']

    if pd.isna(party):
        target_group = grey_group
        tooltip_html = f'<div style="font-family:Arial; padding:8px;"><b>{ward}</b><br><span style="color:#888;">No 2026 election data</span></div>'
        fill_colour = '#cccccc'
        fill_opacity = 0.15
    else:
        council = df[df['Ward'] == ward]['Council'].iloc[0] if ward in df['Ward'].values else None
        target_group = council_groups.get(council, grey_group)

        results = ward_results.get((ward, lad), [])
        max_votes = max(r['Votes'] for r in results) if results else 1

        bars_html = ""
        for r in results:
            c = party_colours.get(r['Party'], '#888888')
            bar_width = int((r['Votes'] / max_votes) * 200)
            bars_html += f"""
            <div style="margin:6px 0; font-size:13px;">
                <div style="margin-bottom:2px;">{r['Party']}: <b>{int(r['Votes'])}</b></div>
                <div style="background:{c}; width:{bar_width}px; height:14px; border-radius:3px;"></div>
            </div>
            """

        note = special_notes.get(ward, '')
        note_html = f'<div style="font-size:11px; color:#888; margin-top:8px;">{note}</div>' if note else ''

        tooltip_html = f"""
        <div style="font-family:Arial; min-width:250px; padding:8px;">
            <b style="font-size:15px;">{ward}</b><br>
            <span style="font-size:12px; color:#888;">{council}</span><br><br>
            {bars_html}
            {note_html}
        </div>
        """
        fill_colour = colour
        fill_opacity = 0.5

    folium.GeoJson(
        row['geometry'],
        style_function=lambda x, fc=fill_colour, fo=fill_opacity: {
            'fillColor': fc,
            'color': 'white',
            'weight': 1.5,
            'fillOpacity': fo
        },
        tooltip=folium.Tooltip(tooltip_html, sticky=True)
    ).add_to(target_group)

# Council boundaries and pie charts -- pie_group only
for council, lad in [('Sheffield', 'Sheffield'), ('Barnsley', 'Barnsley'), ('Wakefield', 'Wakefield')]:
    council_wards = gdf[gdf['LAD22NM'] == lad]
    council_boundary = council_wards.dissolve()

    # Clean subtle boundary -- no dash, no tooltip, no fill
    folium.GeoJson(
        council_boundary,
        style_function=lambda x: {
            'fillColor': 'transparent',
            'color': '#444444',
            'weight': 2,
            'fillOpacity': 0,
            'opacity': 0.6
        }
    ).add_to(pie_group)

    # Pie chart
    composition = council_compositions[council]
    total = sum(composition.values())
    svg = make_pie_svg(composition, party_colours, size=100)

    breakdown = ''.join([
        f'<div style="margin:4px 0; font-size:12px;">'
        f'<span style="background:{party_colours.get(p,"#888")}; width:10px; height:10px; '
        f'display:inline-block; border-radius:2px; margin-right:6px;"></span>'
        f'{p}: <b>{c}</b> seats</div>'
        for p, c in sorted(composition.items(), key=lambda x: -x[1])
    ])

    pie_tooltip = f"""
    <div style="font-family:Arial; min-width:200px; padding:8px;">
        <b style="font-size:15px;">{council}</b><br>
        <span style="font-size:12px; color:#888;">Full council composition</span><br><br>
        {breakdown}
        <div style="font-size:11px; color:#888; margin-top:6px;">Total: {total} seats</div>
    </div>
    """

    icon = folium.DivIcon(
        html=svg,
        icon_size=(100, 100),
        icon_anchor=(50, 50)
    )

    lat, lon = council_centroids[council]
    folium.Marker(
        location=[lat, lon],
        icon=icon,
        tooltip=folium.Tooltip(pie_tooltip, sticky=True)
    ).add_to(pie_group)

# Add all groups in correct order -- remove grey_group
council_groups['Barnsley'].add_to(m)
council_groups['Sheffield'].add_to(m)
council_groups['Wakefield'].add_to(m)
pie_group.add_to(m)

# Layer control
folium.LayerControl(collapsed=False).add_to(m)

# JS opacity toggle with MutationObserver
js_opacity = """
<script>
    setTimeout(function() {
        var checkboxes = document.querySelectorAll('.leaflet-control-layers-overlays input[type=checkbox]');
        var compositionCheckbox = null;
        
        checkboxes.forEach(function(cb) {
            var label = cb.closest('label');
            if (label && label.textContent.trim() === 'Council composition') {
                compositionCheckbox = cb;
            }
        });

        if (!compositionCheckbox) return;

        function isWardPath(path) {
            var fill = path.getAttribute('fill');
            var fillOpacity = parseFloat(path.getAttribute('fill-opacity') || 1);
            if (fill === 'none' || fill === 'transparent' || fillOpacity === 0) return false;
            return true;
        }

        function applyDim() {
            var paths = document.querySelectorAll('.leaflet-overlay-pane svg path');
            paths.forEach(function(path) {
                if (!isWardPath(path)) return;
                path.style.transition = 'fill-opacity 0.6s ease';
                path.style.fillOpacity = 0.25;
            });
        }

        function applyFull() {
            var paths = document.querySelectorAll('.leaflet-overlay-pane svg path');
            paths.forEach(function(path) {
                if (!isWardPath(path)) return;
                path.style.transition = 'fill-opacity 0.6s ease';
                path.style.fillOpacity = 0.5;
            });
        }

        compositionCheckbox.addEventListener('change', function() {
            if (this.checked) {
                applyDim();
            } else {
                applyFull();
            }
        });

        var observer = new MutationObserver(function() {
            if (compositionCheckbox.checked) {
                setTimeout(applyDim, 50);
            }
        });

        var pane = document.querySelector('.leaflet-overlay-pane');
        if (pane) {
            observer.observe(pane, { childList: true, subtree: true });
        }

    }, 2000);
</script>
"""

m.get_root().html.add_child(folium.Element(js_opacity))

# Legend
legend_html = """
<div style="position: fixed; bottom: 0; left: 0; right: 0; z-index: 1000; font-family: Arial;">
    <div style="text-align:center;">
        <button onclick="
            var panel = document.getElementById('legend-panel');
            var btn = document.getElementById('legend-toggle-btn');
            if(panel.style.display === 'none') {
                panel.style.display = 'block';
                btn.innerHTML = '▼ Hide legend';
            } else {
                panel.style.display = 'none';
                btn.innerHTML = '▲ Show legend';
            }"
            id="legend-toggle-btn"
            style="background:white; border:1px solid #ccc; border-radius:6px 6px 0 0;
                   padding:4px 16px; cursor:pointer; font-size:12px;
                   box-shadow: 0 -2px 6px rgba(0,0,0,0.15);">
            ▼ Hide legend
        </button>
    </div>
    <div id="legend-panel" style="background-color: white; padding: 8px 20px;
            box-shadow: 0 -2px 6px rgba(0,0,0,0.2); text-align: center;">
        <div style="white-space: nowrap; overflow-x: auto; font-size: 13px; margin-bottom: 4px;">
            {% for party, colour in parties.items() %}
            <span style="display:inline-block; margin: 0 10px;">
                <span style="background:{{ colour }}; width:12px; height:12px;
                             display:inline-block; border-radius:3px; margin-right:4px;
                             vertical-align:middle;"></span>
                {{ party }}
            </span>
            {% endfor %}
            <span style="display:inline-block; margin: 0 10px;">
                <span style="background:#cccccc; width:12px; height:12px;
                             display:inline-block; border-radius:3px; margin-right:4px;
                             vertical-align:middle;"></span>
                No 2026 data
            </span>
        </div>
        <div style="font-size:11px; color:#555; margin-bottom: 3px;">
            Source: Official results from Sheffield, Barnsley &amp; Wakefield Metropolitan Borough Councils
        </div>
        <div style="font-size:11px; color:#888;">
            Built by encephalon_zest · Co-authored with Claude (Anthropic) · 2026
        </div>
    </div>
</div>
"""

legend = Template(legend_html).render(parties=party_colours)
m.get_root().html.add_child(folium.Element(legend))

OUTPUT = '/home/martin/python_projects/Data_www/Yorkshire_map.html'
m.save(OUTPUT)
print('Done -- refresh your browser!')

Done -- refresh your browser!


In [24]:
js_debug = """
<script>
    setTimeout(function() {
        var paths = document.querySelectorAll('.leaflet-overlay-pane svg path');
        paths.forEach(function(path, i) {
            console.log(
                'Path', i,
                'fill:', path.getAttribute('fill'),
                'fill-opacity:', path.getAttribute('fill-opacity'),
                'stroke:', path.getAttribute('stroke'),
                'style:', path.getAttribute('style')
            );
        });
    }, 3000);
</script>
"""

m.get_root().html.add_child(folium.Element(js_debug))
m.save('/home/martin/python_projects/Data_www/Yorkshire_map.html')
print('Done -- open browser, press F12, go to Console tab, refresh page and paste output here!')

Done -- open browser, press F12, go to Console tab, refresh page and paste output here!


In [25]:
# Updated boundary GeoJSON with custom attribute
for council, lad in [('Sheffield', 'Sheffield'), ('Barnsley', 'Barnsley'), ('Wakefield', 'Wakefield')]:
    council_wards = gdf[gdf['LAD22NM'] == lad]
    council_boundary = council_wards.dissolve()
    council_boundary['boundary'] = 'true'  # tag it

    folium.GeoJson(
        council_boundary,
        style_function=lambda x: {
            'fillColor': 'transparent',
            'color': '#444444',
            'weight': 2,
            'fillOpacity': 0,
            'opacity': 0.6
        },
        name='boundary_line'
    ).add_to(pie_group)

In [29]:
js_opacity = """
<script>
    setTimeout(function() {
        var checkboxes = document.querySelectorAll('.leaflet-control-layers-overlays input[type=checkbox]');
        var compositionCheckbox = null;
        
        checkboxes.forEach(function(cb) {
            var label = cb.closest('label');
            if (label && label.textContent.trim() === 'Council composition') {
                compositionCheckbox = cb;
            }
        });

        if (!compositionCheckbox) return;

        function isWardPath(path) {
            var stroke = path.getAttribute('stroke');
            // Boundary paths have #444444 stroke -- skip them
            if (stroke === '#444444') return false;
            var fillOpacity = parseFloat(path.getAttribute('fill-opacity') || 0);
            if (fillOpacity === 0) return false;
            return true;
        }

        function applyDim() {
            var paths = document.querySelectorAll('.leaflet-overlay-pane svg path');
            paths.forEach(function(path) {
                if (!isWardPath(path)) return;
                path.style.transition = 'fill-opacity 0.6s ease';
                path.style.fillOpacity = 0.25;
            });
        }

        function applyFull() {
            var paths = document.querySelectorAll('.leaflet-overlay-pane svg path');
            paths.forEach(function(path) {
                if (!isWardPath(path)) return;
                path.style.transition = 'fill-opacity 0.6s ease';
                path.style.fillOpacity = 0.5;
            });
        }

        compositionCheckbox.addEventListener('change', function() {
            if (this.checked) {
                applyDim();
            } else {
                applyFull();
            }
        });

        var observer = new MutationObserver(function() {
            if (compositionCheckbox.checked) {
                setTimeout(applyDim, 50);
            }
        });

        var pane = document.querySelector('.leaflet-overlay-pane');
        if (pane) {
            observer.observe(pane, { childList: true, subtree: true });
        }

    }, 2000);
</script>
"""

m.get_root().html.add_child(folium.Element(js_opacity))

OUTPUT = '/home/martin/python_projects/Data_www/Yorkshire_map.html'
m.save(OUTPUT)
print('Done -- refresh and test!')

Done -- refresh and test!


In [27]:
boundary_group = folium.FeatureGroup(name='Council boundaries', show=False)

for council, lad in [('Sheffield', 'Sheffield'), ('Barnsley', 'Barnsley'), ('Wakefield', 'Wakefield')]:
    council_wards = gdf[gdf['LAD22NM'] == lad]
    council_boundary = council_wards.dissolve()

    folium.GeoJson(
        council_boundary,
        style_function=lambda x: {
            'fillColor': 'transparent',
            'color': '#444444',
            'weight': 2,
            'fillOpacity': 0,
            'opacity': 0.6
        }
    ).add_to(boundary_group)

In [30]:
council_groups['Barnsley'].add_to(m)
council_groups['Sheffield'].add_to(m)
council_groups['Wakefield'].add_to(m)
boundary_group.add_to(m)
pie_group.add_to(m)
folium.LayerControl(collapsed=False).add_to(m)

In [31]:
# Replace your base map line with this
m = folium.Map(location=[53.6, -1.5], zoom_start=10)

# Add base tile explicitly as a named layer
folium.TileLayer('CartoDB positron', name='Base map').add_to(m)

In [32]:
# Special notes
special_notes = {
    'Firth Park': '⚠️ Two seats contested — Labour Party & Reform UK each won one seat.',
    'Wakefield East': '⚠️ Three seats — Labour Party 1, Reform UK 2.',
    'Cudworth': '⚠️ Three seats — Reform UK 1, Labour Party 2.',
    'Penistone East': '⚠️ Three seats — Labour Party 2, Reform UK 1.',
    'Penistone West': '⚠️ Three seats — Liberal Democrats 2, Reform UK 1.',
    'Rockingham': '⚠️ Three seats — Reform UK 2, Independent 1.',
    'Knottingley': '⚠️ Three seats — Reform UK 1, Liberal Democrats 2.',
    'Wakefield North': '⚠️ Three seats — Green Party 1, Reform UK 2.',
    'Wakefield South': '⚠️ Three seats — Conservative Party 1, Reform UK 2.',
    'Wombwell': '⚠️ Three seats — Reform UK 2, Labour Party 1.',
    'Worsbrough': '⚠️ Three seats — Reform UK 2, Independent 1.'
}

# Base map with explicit tile layer
m = folium.Map(location=[53.6, -1.5], zoom_start=10)
folium.TileLayer('CartoDB positron', name='Base map').add_to(m)

# Inject CSS
meta = Element("""
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<style>
    .leaflet-tooltip {
        font-size: 14px !important;
        min-width: 260px !important;
        padding: 10px !important;
        pointer-events: none;
    }
</style>
""")
m.get_root().header.add_child(meta)

# FeatureGroups
councils = ['Sheffield', 'Barnsley', 'Wakefield']
council_groups = {}
for council in councils:
    council_groups[council] = folium.FeatureGroup(name=council, show=True)

boundary_group = folium.FeatureGroup(name='Council boundaries', show=False)
pie_group = folium.FeatureGroup(name='Council composition', show=False)

# Ward polygons
for _, row in gdf_merged.iterrows():
    ward = row['Ward']
    lad = row['LAD22NM']
    party = row['Party']
    colour = row['colour']

    if pd.isna(party):
        target_group = council_groups.get('Sheffield', None)
        tooltip_html = f'<div style="font-family:Arial; padding:8px;"><b>{ward}</b><br><span style="color:#888;">No 2026 election data</span></div>'
        fill_colour = '#cccccc'
        fill_opacity = 0.15
        target_group = folium.FeatureGroup(name='dummy', show=False)
    else:
        council = df[df['Ward'] == ward]['Council'].iloc[0] if ward in df['Ward'].values else None
        target_group = council_groups.get(council, council_groups['Sheffield'])

        results = ward_results.get((ward, lad), [])
        max_votes = max(r['Votes'] for r in results) if results else 1

        bars_html = ""
        for r in results:
            c = party_colours.get(r['Party'], '#888888')
            bar_width = int((r['Votes'] / max_votes) * 200)
            bars_html += f"""
            <div style="margin:6px 0; font-size:13px;">
                <div style="margin-bottom:2px;">{r['Party']}: <b>{int(r['Votes'])}</b></div>
                <div style="background:{c}; width:{bar_width}px; height:14px; border-radius:3px;"></div>
            </div>
            """

        note = special_notes.get(ward, '')
        note_html = f'<div style="font-size:11px; color:#888; margin-top:8px;">{note}</div>' if note else ''

        tooltip_html = f"""
        <div style="font-family:Arial; min-width:250px; padding:8px;">
            <b style="font-size:15px;">{ward}</b><br>
            <span style="font-size:12px; color:#888;">{council}</span><br><br>
            {bars_html}
            {note_html}
        </div>
        """
        fill_colour = colour
        fill_opacity = 0.5

    folium.GeoJson(
        row['geometry'],
        style_function=lambda x, fc=fill_colour, fo=fill_opacity: {
            'fillColor': fc,
            'color': 'white',
            'weight': 1.5,
            'fillOpacity': fo
        },
        tooltip=folium.Tooltip(tooltip_html, sticky=True)
    ).add_to(target_group)

# Council boundaries -- own independent group
for council, lad in [('Sheffield', 'Sheffield'), ('Barnsley', 'Barnsley'), ('Wakefield', 'Wakefield')]:
    council_wards = gdf[gdf['LAD22NM'] == lad]
    council_boundary = council_wards.dissolve()

    folium.GeoJson(
        council_boundary,
        style_function=lambda x: {
            'fillColor': 'transparent',
            'color': '#444444',
            'weight': 2,
            'fillOpacity': 0,
            'opacity': 0.6
        }
    ).add_to(boundary_group)

# Pie charts -- pie_group only
for council, lad in [('Sheffield', 'Sheffield'), ('Barnsley', 'Barnsley'), ('Wakefield', 'Wakefield')]:
    composition = council_compositions[council]
    total = sum(composition.values())
    svg = make_pie_svg(composition, party_colours, size=100)

    breakdown = ''.join([
        f'<div style="margin:4px 0; font-size:12px;">'
        f'<span style="background:{party_colours.get(p,"#888")}; width:10px; height:10px; '
        f'display:inline-block; border-radius:2px; margin-right:6px;"></span>'
        f'{p}: <b>{c}</b> seats</div>'
        for p, c in sorted(composition.items(), key=lambda x: -x[1])
    ])

    pie_tooltip = f"""
    <div style="font-family:Arial; min-width:200px; padding:8px;">
        <b style="font-size:15px;">{council}</b><br>
        <span style="font-size:12px; color:#888;">Full council composition</span><br><br>
        {breakdown}
        <div style="font-size:11px; color:#888; margin-top:6px;">Total: {total} seats</div>
    </div>
    """

    icon = folium.DivIcon(
        html=svg,
        icon_size=(100, 100),
        icon_anchor=(50, 50)
    )

    lat, lon = council_centroids[council]
    folium.Marker(
        location=[lat, lon],
        icon=icon,
        tooltip=folium.Tooltip(pie_tooltip, sticky=True)
    ).add_to(pie_group)

# Add all groups in order
council_groups['Barnsley'].add_to(m)
council_groups['Sheffield'].add_to(m)
council_groups['Wakefield'].add_to(m)
boundary_group.add_to(m)
pie_group.add_to(m)

# Layer control
folium.LayerControl(collapsed=False).add_to(m)

# JS opacity toggle
js_opacity = """
<script>
    setTimeout(function() {
        var checkboxes = document.querySelectorAll('.leaflet-control-layers-overlays input[type=checkbox]');
        var compositionCheckbox = null;
        
        checkboxes.forEach(function(cb) {
            var label = cb.closest('label');
            if (label && label.textContent.trim() === 'Council composition') {
                compositionCheckbox = cb;
            }
        });

        if (!compositionCheckbox) return;

        function isWardPath(path) {
            var stroke = path.getAttribute('stroke');
            if (stroke === '#444444') return false;
            var fillOpacity = parseFloat(path.getAttribute('fill-opacity') || 0);
            if (fillOpacity === 0) return false;
            return true;
        }

        function applyDim() {
            var paths = document.querySelectorAll('.leaflet-overlay-pane svg path');
            paths.forEach(function(path) {
                if (!isWardPath(path)) return;
                path.style.transition = 'fill-opacity 0.6s ease';
                path.style.fillOpacity = 0.25;
            });
        }

        function applyFull() {
            var paths = document.querySelectorAll('.leaflet-overlay-pane svg path');
            paths.forEach(function(path) {
                if (!isWardPath(path)) return;
                path.style.transition = 'fill-opacity 0.6s ease';
                path.style.fillOpacity = 0.5;
            });
        }

        compositionCheckbox.addEventListener('change', function() {
            if (this.checked) {
                applyDim();
            } else {
                applyFull();
            }
        });

        var observer = new MutationObserver(function() {
            if (compositionCheckbox.checked) {
                setTimeout(applyDim, 50);
            }
        });

        var pane = document.querySelector('.leaflet-overlay-pane');
        if (pane) {
            observer.observe(pane, { childList: true, subtree: true });
        }

    }, 2000);
</script>
"""

m.get_root().html.add_child(folium.Element(js_opacity))

# Legend
legend_html = """
<div style="position: fixed; bottom: 0; left: 0; right: 0; z-index: 1000; font-family: Arial;">
    <div style="text-align:center;">
        <button onclick="
            var panel = document.getElementById('legend-panel');
            var btn = document.getElementById('legend-toggle-btn');
            if(panel.style.display === 'none') {
                panel.style.display = 'block';
                btn.innerHTML = '▼ Hide legend';
            } else {
                panel.style.display = 'none';
                btn.innerHTML = '▲ Show legend';
            }"
            id="legend-toggle-btn"
            style="background:white; border:1px solid #ccc; border-radius:6px 6px 0 0;
                   padding:4px 16px; cursor:pointer; font-size:12px;
                   box-shadow: 0 -2px 6px rgba(0,0,0,0.15);">
            ▼ Hide legend
        </button>
    </div>
    <div id="legend-panel" style="background-color: white; padding: 8px 20px;
            box-shadow: 0 -2px 6px rgba(0,0,0,0.2); text-align: center;">
        <div style="white-space: nowrap; overflow-x: auto; font-size: 13px; margin-bottom: 4px;">
            {% for party, colour in parties.items() %}
            <span style="display:inline-block; margin: 0 10px;">
                <span style="background:{{ colour }}; width:12px; height:12px;
                             display:inline-block; border-radius:3px; margin-right:4px;
                             vertical-align:middle;"></span>
                {{ party }}
            </span>
            {% endfor %}
            <span style="display:inline-block; margin: 0 10px;">
                <span style="background:#cccccc; width:12px; height:12px;
                             display:inline-block; border-radius:3px; margin-right:4px;
                             vertical-align:middle;"></span>
                No 2026 data
            </span>
        </div>
        <div style="font-size:11px; color:#555; margin-bottom: 3px;">
            Source: Official results from Sheffield, Barnsley &amp; Wakefield Metropolitan Borough Councils
        </div>
        <div style="font-size:11px; color:#888;">
            Built by encephalon_zest · Co-authored with Claude (Anthropic) · 2026
        </div>
    </div>
</div>
"""

legend = Template(legend_html).render(parties=party_colours)
m.get_root().html.add_child(folium.Element(legend))

OUTPUT = '/home/martin/python_projects/Data_www/Yorkshire_map.html'
m.save(OUTPUT)
print('Done -- refresh your browser!')

Done -- refresh your browser!
